# Ablation Study Overview: S1–S7 Retrieval Systems

*(Notebook: `AblationStudy_S1-7.ipynb`)*

This Notebook is the built for the 7-system ablation study, which was designed to build and validate the **ComplianceGPT Retriever** architecture.  

---

## 2. Final Consolidated Results (MRR@10)

This table summarizes the final performance of all 7 systems across all 6 evaluation datasets (Overall, ODP-Only, and Error Bank).

| System | Description | Rev. 5 (Overall) | Rev. 5 (ODP) | Rev. 4 (Overall) | Rev. 4 (ODP) | Error Bank (Rev. 5) | Error Bank (Rev. 4) |
|---|---|---:|---:|---:|---:|---:|---:|
| S1 | BM25 (Baseline) | 0.8512 | 0.8907 | 0.7239 | 0.8246 | 0.3966 | 0.3690 |
| S2 | Dense (Baseline) | 0.8524 | 0.8806 | 0.8472 | 0.7719 | 0.7399 | 0.7879 |
| S3 | Rewrite-Only | 0.6085 | 0.6400 | 0.6401 | 0.6813 | 0.2604 | 0.3948 |
| S4 | QUR-RRF | 0.7802 | 0.7780 | 0.8114 | 0.8947 | 0.4493 | 0.5586 |
| S5 | Hybrid RRF (S1+S2) | 0.9153 | 0.9286 | 0.8125 | 0.8684 | 0.7152 | 0.5985 |
| S6 | Hybrid + Rerank | 0.9398 | 0.9497 | 0.9444 | 0.9474 | 0.8106 | 0.8182 |
| S7 | ComplianceGPT | 0.9448 | 0.9577 | 0.9444 | 0.9474 | 0.8323 | 0.8182 |

---

## 3. The 7‑System Narrative: What Each System Proved

- **S1 (BM25 Baseline):** Proved that a simple lexical retriever is a strong baseline but fails on semantic/terminology gaps (low Error Bank score).  
- **S2 (Dense Baseline):** Proved that a semantic retriever is complementary to S1, performing much better on the “hard” Rev. 4 and Error Bank queries.  
- **S3 (Rewrite‑Only):** Proved that naively replacing a query with its semantic rewrite and feeding it to a lexical (BM25) retriever is a failed strategy. This justifies the need for fusion.  
- **S4 (QUR‑RRF):** Proved that fusion (RRF) is the correct way to use query rewrites with BM25, creating a system that beats the S1 baseline on harder (Rev. 4) queries.  
- **S5 (Hybrid RRF):** Proved that fusing the two baselines (S1+S2) creates a superior high‑recall retriever that outperforms all previous systems on the main (Rev. 5) dataset.  
- **S6 (Hybrid + Rerank):** Proved that adding a reranker to S5’s high‑recall list is the key to precision, solving the “noise problem” from S5 and achieving state‑of‑the‑art results on the Error Bank.  
- **S7 (ComplianceGPT):** Proved that the full architecture (QUR + Hybrid + Reranker) is the optimal system, as the QUR component provides a final, measurable boost on top of S6, especially on the Error Bank.

---

## 4. System READMEs

For a detailed breakdown of each experiment, see the individual READMEs:

- `README_S1.md`  
- `README_S2.md`  
- `README_S3.md`  
- `README_S4.md`  
- `README_S5.md`  
- `README_S6.md`  
- `README_S7.md`


# README: System 1 — BM25 Baseline
*(Notebook: `AblationStudy_S1-7.ipynb`)*

## 1. Overview
This README describes **System 1 (BM25 Baseline)** for the ComplianceGPT retrieval experiments.

Its primary purpose is to establish the baseline performance of a standard **BM25 (Okapi)** retriever against the **NIST SP 800-53 Rev. 5** and **Rev. 4** gold standard query sets. The performance metrics generated by this notebook serve as the benchmark against which all other, more advanced systems (Systems 2–7) will be compared.

This notebook also serves two functions related to the **Error Bank**:
1. **Identification:** The detailed output CSVs (`S1_bm25_rev5_results.csv`, `S1_bm25_rev4_results.csv`) are used to *manually identify* the queries that the BM25 baseline fails to answer correctly.
2. **Evaluation:** The notebook contains an optional section to run a formal evaluation *against* this Error Bank, confirming its difficulty (e.g., a **Recall@1 = 0.0**).

## 2. Setup & Requirements
Execution of the notebook has the following requirements:

- **Google Drive:** The notebook is designed to be run in an environment with Google Drive mounted (e.g., Google Colab).
- **Corpus Files:** The `CONFIG` cell (Cell 3) must be configured with the correct paths for:
  - `NIST_SP-800-53_rev5_catalog.jsonl`
  - `NIST_SP-800-53_rev4_catalog.jsonl`
- **Gold Set Files:** The `CONFIG` cell must also contain paths to:
  - `nist_sp800-53_rev5_gold-set_100q.csv`
  - `nist_sp800-53_rev4_gold-set_36q.csv`

## 3. Main Workflow (Baseline Run)
This workflow generates the main baseline metrics and detailed results.

1. **Mount Drive:** The first cell mounts the Google Drive.
2. **Execute Cells:** Execution of all cells down to the **MAIN** cell (ID: `d874fa44`) performs the following operations:
   - Loads the corpora and gold sets.
   - Tokenizes the corpora and builds the BM25 models (`bm25_rev5`, `bm25_rev4`).
   - Runs the evaluation for all Rev. 5 (100q) and Rev. 4 (36q) questions.
   - Prints the 4 main summary tables (Rev5 Overall/ODP, Rev4 Overall/ODP).
   - Saves the detailed CSVs and summary report to the `/content/` folder.
3. **Save to Drive:** The final cell (**“Save to Drive”**) is executed to permanently move all generated outputs to the Google Drive folder:
   `/content/drive/MyDrive/ablation_outputs/system_1/`

## 4. Error Bank Workflow
This is a two-part process: (A) Manual creation of the error bank, and (B) Evaluation of the error bank.

### A. Error Bank Creation Process
1. **Execute Main Workflow:** All steps in Section 3 are completed.
2. **Inspect Results:** The `S1_bm25_rev5_results.csv` and `S1_bm25_rev4_results.csv` files from the Google Drive output folder are inspected.
3. **Identify Failures:** The CSVs are filtered/sorted to find all queries where the `rank` is **greater than 1** or is **0/NaN** (indicating the correct control was not found in the Top 10).
4. **Build New CSV:** These failed queries are copied into a new local CSV file. This file **must** include:
   - `question`
   - `control_id`
   - `odp_required`
   - A new column: **`version`** (with values `rev5` or `rev4`).
5. **Save:** This new file is saved as `error_bank_v1.csv`.

### B. Error Bank Evaluation Process
With `error_bank_v1.csv` available, the baseline evaluation can be run against it.

1. **Upload:** In Google Colab, the `error_bank_v1.csv` file is uploaded to the `/content/` directory.
2. **Execute Cell 1:** The **“Optional: Error Bank Re-evaluation”** cell (ID: `d1421715`) is re-executed. This loads the uploaded CSV, splits it by version, and prints the two Error Bank summary tables.
3. **Execute Cell 2:** Re-executing the final **“Save to Drive”** cell moves the newly created `S1_bm25_error_bank_...` files to Google Drive, overwriting any old versions.

## 5. Outputs
This notebook generates 5 key result files, saved to:
`/content/drive/MyDrive/ablation_outputs/system_1/`

- `S1_bm25_rev5_results.csv`: Per-query results for the 100 Rev. 5 questions.
- `S1_bm25_rev4_results.csv`: Per-query results for the 36 Rev. 4 questions.
- `S1_bm25_report.md`: A markdown file containing the 4 main summary tables (Rev5/Rev4, Overall/ODP).
- `S1_bm25_error_bank_rev5_results.csv`: *(Optional)* Detailed results for the Rev. 5 error bank questions.
- `S1_bm25_error_bank_rev4_results.csv`: *(Optional)* Detailed results for the Rev. 4 error bank questions.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

# === Config ===
REV5_CATALOG_PATH = "/content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl"
REV4_CATALOG_PATH = "/content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl"

REV5_GOLD_CSV = "/content/drive/MyDrive/compliance_data/gold_standard_datasets/nist800_53/nist_sp800-53_rev5_gold-set_100q.csv"
REV4_GOLD_CSV = "/content/drive/MyDrive/compliance_data/gold_standard_datasets/nist800_53/nist_sp800-53_rev4_gold-set_36q.csv"

# error bank path
ERROR_BANK_CSV = "/content/drive/MyDrive/compliance_data/error_bank/error_bank_v1.csv"

REV5_OUT = "S1_bm25_rev5_results.csv"
REV4_OUT = "S1_bm25_rev4_results.csv"

SUMMARY_MD = "S1_bm25_report.md"

TOP_N = 10
BM25_K1 = 1.5
BM25_B = 0.75


In [3]:

import json, math, re, pandas as pd, numpy as np, os
from typing import List, Dict, Tuple, Optional
pd.set_option("display.max_colwidth", 200)


In [4]:

# ------------------------------
# Normalization & tokenization
# ------------------------------

PUNCT_RE = re.compile(r"[^0-9A-Za-z_\s]+")  # remove basic punctuation
WS_RE = re.compile(r"\s+")
TOKEN_RE = re.compile(r"[A-Za-z0-9]+")

def normalize_text(text: str) -> str:
    """Lowercase, remove punctuation (keep letters/numbers/_), collapse whitespace."""
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)
    s = text.lower()
    # hyphen becomes space so 'ac-3' -> 'ac 3'
    s = s.replace('-', ' ')
    s = PUNCT_RE.sub(' ', s)
    s = WS_RE.sub(' ', s).strip()
    return s

def tokenize(text: str) -> List[str]:
    """Alphanumeric tokens, lowercased (expects normalized text)."""
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)
    return [t.lower() for t in TOKEN_RE.findall(text)]


In [5]:

# ------------------------------
# CCS parsing: extract control_id + text
# ------------------------------

def extract_text_from_record(rec: Dict) -> str:
    texts: List[str] = []

    def recurse(obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                if k == "prose" and isinstance(v, str):
                    texts.append(v)
                else:
                    recurse(v)
        elif isinstance(obj, list):
            for it in obj:
                recurse(it)

    # top-level title/prose
    if isinstance(rec.get("title"), str):
        texts.append(rec["title"])
    if isinstance(rec.get("prose"), str):
        texts.append(rec["prose"])

    # parts tree
    recurse(rec.get("parts", []))

    # params guidelines (ODP notes)
    for prm in rec.get("params") or []:
        if isinstance(prm, dict):
            recurse(prm.get("guidelines", []))

    # join
    return "\n".join(texts)

def extract_control_id(rec: Dict) -> Optional[str]:
    candidates = []
    for prop in rec.get("props", []) or []:
        if isinstance(prop, dict) and prop.get("name") == "label":
            val = prop.get("value")
            if isinstance(val, str) and val.strip():
                candidates.append(val.strip())

    if candidates:
        def rank(v: str):
            has_paren = '(' in v
            zero_padded = bool(re.search(r"-0\d", v))
            return (has_paren, not zero_padded, len(v))
        candidates.sort(key=rank, reverse=True)
        return candidates[0]

    rid = rec.get("id")
    return rid.upper() if isinstance(rid, str) else None

def control_id_aliases(cid: str) -> List[str]:
    """Variants of control_id to help match ID mentions in queries."""
    cid_clean = cid.strip()
    return [
        cid_clean,                                # AC-3
        cid_clean.replace('-', ' '),              # AC 3
        cid_clean.replace('-', ''),               # AC3
    ]

def load_corpus_jsonl(path: str) -> List[Dict[str, str]]:
    docs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            cid = extract_control_id(rec)
            if not cid:
                continue
            body = extract_text_from_record(rec)
            if not isinstance(body, str) or not body.strip():
                continue
            # Prefix control_id variants to aid ID-based queries
            prefix = " ".join(control_id_aliases(cid))
            full_text = prefix + "\n" + body
            docs.append({"control_id": cid, "text": full_text})
    return docs


In [6]:

# ------------------------------
# Minimal BM25 (with precomputed TF)
# ------------------------------

class BM25Okapi:
    def __init__(self, corpus_tokens: List[List[str]], k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.corpus_tokens = corpus_tokens
        self.N = len(corpus_tokens)
        self.doc_len = np.array([len(d) for d in corpus_tokens], dtype=np.float32)
        self.avgdl = float(self.doc_len.mean()) if self.N else 0.0

        # term freq per doc and doc freq
        self.tf: List[Dict[str, int]] = []
        df: Dict[str, int] = {}
        for doc in corpus_tokens:
            tf_doc: Dict[str, int] = {}
            for tok in doc:
                tf_doc[tok] = tf_doc.get(tok, 0) + 1
            self.tf.append(tf_doc)
            for term in tf_doc.keys():
                df[term] = df.get(term, 0) + 1

        # BM25+ style IDF (avoids negative idf)
        self.idf = {term: math.log((self.N - freq + 0.5) / (freq + 0.5) + 1.0) for term, freq in df.items()}

    def get_scores(self, query_tokens: List[str]) -> np.ndarray:
        scores = np.zeros(self.N, dtype=np.float32)
        for term in query_tokens:
            idf_t = self.idf.get(term)
            if idf_t is None:
                continue
            for i, tf_doc in enumerate(self.tf):
                f = tf_doc.get(term, 0)
                if f == 0:
                    continue
                denom = f + self.k1 * (1 - self.b + self.b * (self.doc_len[i] / (self.avgdl + 1e-9)))
                scores[i] += idf_t * (f * (self.k1 + 1)) / (denom + 1e-9)
        return scores

    def get_top_n(self, query_tokens: List[str], n: int = 10) -> List[int]:
        scores = self.get_scores(query_tokens)
        if n >= len(scores):
            return list(np.argsort(-scores))
        idx = np.argpartition(-scores, n-1)[:n]
        idx = idx[np.argsort(-scores[idx])]
        return list(idx)


In [7]:

# ------------------------------
# Metrics (rank==0 treated as miss)
# ------------------------------

def _to_int_rank(x) -> int:
    try:
        if pd.isna(x): return 0
    except Exception:
        pass
    try:
        r = int(x)
        return r if r >= 0 else 0
    except Exception:
        return 0

def recall_at_k(ranks: List[int], k: int) -> float:
    ranks = [_to_int_rank(r) for r in ranks]
    if not ranks: return 0.0
    hits = sum(1 for r in ranks if 1 <= r <= k)
    return hits / len(ranks)

def mrr_at_k(ranks: List[int], k: int) -> float:
    ranks = [_to_int_rank(r) for r in ranks]
    if not ranks: return 0.0
    rrs = [(1.0 / r) if (1 <= r <= k) else 0.0 for r in ranks]
    return float(np.mean(rrs)) if rrs else 0.0

def ndcg_at_k(ranks: List[int], k: int) -> float:
    ranks = [_to_int_rank(r) for r in ranks]
    if not ranks: return 0.0
    vals = [(1.0 / math.log2(r + 1.0)) if (1 <= r <= k) else 0.0 for r in ranks]
    return float(np.mean(vals)) if vals else 0.0

def summarize_metrics(ranks: List[int], k_values=(1,5,10)) -> pd.DataFrame:
    rows = []
    for k in k_values:
        rows.append({
            "k": k,
            "Recall@k": recall_at_k(ranks, k),
            "MRR@k": mrr_at_k(ranks, k),
            "nDCG@k": ndcg_at_k(ranks, k),
        })
    df = pd.DataFrame(rows)[["k","Recall@k","MRR@k","nDCG@k"]]
    return df


In [8]:

# ------------------------------
# Build BM25 + evaluate gold
# ------------------------------

def build_bm25_from_corpus(docs: List[Dict[str, str]], k1=BM25_K1, b=BM25_B) -> Tuple[BM25Okapi, List[str]]:
    control_ids = [d["control_id"] for d in docs]
    tokens = [tokenize(normalize_text(d["text"])) for d in docs]
    bm25 = BM25Okapi(tokens, k1=k1, b=b)
    return bm25, control_ids

def evaluate_gold(gold_df: pd.DataFrame, bm25: BM25Okapi, control_ids: List[str], top_n: int = TOP_N) -> pd.DataFrame:
    records = []
    for _, row in gold_df.iterrows():
        q = str(row.get("question", "")).strip()
        gold_cid = str(row.get("control_id", "")).strip()
        odp_req = str(row.get("odp_required", "N/A")).strip()

        q_norm = normalize_text(q)
        q_tokens = tokenize(q_norm)

        idxs = bm25.get_top_n(q_tokens, n=top_n)
        retrieved = [control_ids[i] for i in idxs]

        # 1-based rank or 0 if not found
        rank = 0
        for pos, cid in enumerate(retrieved, start=1):
            if cid == gold_cid:
                rank = pos
                break

        records.append({
            "question": q,
            "gold_control_id": gold_cid,
            "odp_required": odp_req,
            "retrieved_control_ids": retrieved,
            "rank": rank
        })
    return pd.DataFrame(records)


In [9]:
# ------------------------------
# Final reporting (display + save)
# ------------------------------

def _rank_list(df: pd.DataFrame) -> List[int]:
    return [int(r) if pd.notna(r) else 0 for r in df["rank"].tolist()]

def _odp_mask(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip()
    up = s.str.upper()
    return ~(s.eq("") | up.isin(["N/A", "NA", "NONE", "NULL", "NAN"]))

def display_summary_report(results_df: pd.DataFrame, version_name: str):
    ranks_all = _rank_list(results_df)
    n_all = len(results_df)

    mask_odp = _odp_mask(results_df["odp_required"])
    odp_df = results_df[mask_odp].copy()
    ranks_odp = _rank_list(odp_df)
    n_odp = len(odp_df)

    # Compute metrics
    r1_all = recall_at_k(ranks_all, 1); r5_all = recall_at_k(ranks_all, 5); r10_all = recall_at_k(ranks_all, 10)
    mrr_all = mrr_at_k(ranks_all, 10); ndcg_all = ndcg_at_k(ranks_all, 10)

    r1_odp = recall_at_k(ranks_odp, 1); r5_odp = recall_at_k(ranks_odp, 5); r10_odp = recall_at_k(ranks_odp, 10)
    mrr_odp = mrr_at_k(ranks_odp, 10); ndcg_odp = ndcg_at_k(ranks_odp, 10)

    # Print header and table
    print(f"\n--- {version_name}: Performance ---")
    print(f"| Metric    | Overall ({n_all}q) | ODP-Subset ({n_odp}q) |")
    print("|-----------|------------------|--------------------|")
    print(f"| Recall@1  | {r1_all:.4f}           | {r1_odp:.4f}             |")
    print(f"| Recall@5  | {r5_all:.4f}           | {r5_odp:.4f}             |")
    print(f"| Recall@10 | {r10_all:.4f}           | {r10_odp:.4f}             |")
    print(f"| MRR@10    | {mrr_all:.4f}           | {mrr_odp:.4f}             |")
    print(f"| nDCG@10   | {ndcg_all:.4f}           | {ndcg_odp:.4f}             |")

def _compute_summary_values(results_df: pd.DataFrame):
    ranks_all = _rank_list(results_df)
    n_all = len(results_df)
    mask_odp = _odp_mask(results_df["odp_required"])
    odp_df = results_df[mask_odp].copy()
    ranks_odp = _rank_list(odp_df)
    n_odp = len(odp_df)

    return {
        "n_all": n_all, "n_odp": n_odp,
        "r1_all": recall_at_k(ranks_all, 1), "r5_all": recall_at_k(ranks_all, 5), "r10_all": recall_at_k(ranks_all, 10),
        "mrr_all": mrr_at_k(ranks_all, 10), "ndcg_all": ndcg_at_k(ranks_all, 10),
        "r1_odp": recall_at_k(ranks_odp, 1), "r5_odp": recall_at_k(ranks_odp, 5), "r10_odp": recall_at_k(ranks_odp, 10),
        "mrr_odp": mrr_at_k(ranks_odp, 10), "ndcg_odp": ndcg_at_k(ranks_odp, 10),
    }

def save_summary_report(
    rev5_df: pd.DataFrame,
    rev4_df: pd.DataFrame,
    out_path: str,
    system_name: str,
    err_rev5_df: Optional[pd.DataFrame] = None,
    err_rev4_df: Optional[pd.DataFrame] = None
):
    v5 = _compute_summary_values(rev5_df)
    v4 = _compute_summary_values(rev4_df)

    # Build the exact-format markdown string (tabs for columns)
    lines = []
    lines = []
    lines.append(f"{system_name}: Performance Evaluation")
    lines.append(f"This report summarizes the performance of the {system_name.lower()} on the NIST SP 800-53 Rev. 5 and Rev. 4 gold sets, and error bank set.")
    lines.append("")
    lines.append("NIST SP 800-53 Rev. 5: Performance")
    lines.append(f"Metric\tOverall ({v5['n_all']}q)\tODP-Subset ({v5['n_odp']}q)")
    lines.append(f"Recall@1\t{v5['r1_all']:.4f}\t{v5['r1_odp']:.4f}")
    lines.append(f"Recall@5\t{v5['r5_all']:.4f}\t{v5['r5_odp']:.4f}")
    lines.append(f"Recall@10\t{v5['r10_all']:.4f}\t{v5['r10_odp']:.4f}")
    lines.append(f"MRR@10\t{v5['mrr_all']:.4f}\t{v5['mrr_odp']:.4f}")
    lines.append(f"nDCG@10\t{v5['ndcg_all']:.4f}\t{v5['ndcg_odp']:.4f}")
    lines.append("")
    lines.append("NIST SP 800-53 Rev. 4: Performance")
    lines.append(f"Metric\tOverall ({v4['n_all']}q)\tODP-Subset ({v4['n_odp']}q)")
    lines.append(f"Recall@1\t{v4['r1_all']:.4f}\t{v4['r1_odp']:.4f}")
    lines.append(f"Recall@5\t{v4['r5_all']:.4f}\t{v4['r5_odp']:.4f}")
    lines.append(f"Recall@10\t{v4['r10_all']:.4f}\t{v4['r10_odp']:.4f}")
    lines.append(f"MRR@10\t{v4['mrr_all']:.4f}\t{v4['mrr_odp']:.4f}")
    lines.append(f"nDCG@10\t{v4['ndcg_all']:.4f}\t{v4['ndcg_odp']:.4f}")

    # --- Add Error Bank Sections (if provided) ---
    if err_rev5_df is not None and not err_rev5_df.empty:
        v5_err = _compute_summary_values(err_rev5_df)
        lines.append("")
        lines.append(f"Error Bank — Rev.5 only (n={v5_err['n_all']})")
        lines.append(f"Metric\tOverall ({v5_err['n_all']}q)\tODP-Subset ({v5_err['n_odp']}q)")
        lines.append(f"Recall@1\t{v5_err['r1_all']:.4f}\t{v5_err['r1_odp']:.4f}")
        lines.append(f"Recall@5\t{v5_err['r5_all']:.4f}\t{v5_err['r5_odp']:.4f}")
        lines.append(f"Recall@10\t{v5_err['r10_all']:.4f}\t{v5_err['r10_odp']:.4f}")
        lines.append(f"MRR@10\t{v5_err['mrr_all']:.4f}\t{v5_err['mrr_odp']:.4f}")
        lines.append(f"nDCG@10\t{v5_err['ndcg_all']:.4f}\t{v5_err['ndcg_odp']:.4f}")

    if err_rev4_df is not None and not err_rev4_df.empty:
        v4_err = _compute_summary_values(err_rev4_df)
        lines.append("")
        lines.append(f"Error Bank — Rev.4 only (n={v4_err['n_all']})")
        lines.append(f"Metric\tOverall ({v4_err['n_all']}q)\tODP-Subset ({v4_err['n_odp']}q)")
        lines.append(f"Recall@1\t{v4_err['r1_all']:.4f}\t{v4_err['r1_odp']:.4f}")
        lines.append(f"Recall@5\t{v4_err['r5_all']:.4f}\t{v4_err['r5_odp']:.4f}")
        lines.append(f"Recall@10\t{v4_err['r10_all']:.4f}\t{v4_err['r10_odp']:.4f}")
        lines.append(f"MRR@10\t{v4_err['mrr_all']:.4f}\t{v4_err['mrr_odp']:.4f}")
        lines.append(f"nDCG@10\t{v4_err['ndcg_all']:.4f}\t{v4_err['ndcg_odp']:.4f}")
    # --- End of Error Bank addition ---

    text = "\n".join(lines)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"Saved summary report -> {os.path.abspath(out_path)}")

In [10]:

# ------------------------------
# MAIN: load, clean, evaluate, save, summarize, and export report
# ------------------------------

# Load gold sets (allow ISO-8859-1 just in case)
rev5_gold = pd.read_csv(REV5_GOLD_CSV, encoding="ISO-8859-1")
rev4_gold = pd.read_csv(REV4_GOLD_CSV, encoding="ISO-8859-1")

# Data cleaning
rev5_gold.dropna(subset=["question"], inplace=True)
rev4_gold.dropna(subset=["question"], inplace=True)
rev5_gold["question"] = rev5_gold["question"].astype(str).str.strip().str.lstrip(": ")
rev4_gold["question"] = rev4_gold["question"].astype(str).str.strip().str.lstrip(": ")

print("Loading corpora ......")
rev5_docs = load_corpus_jsonl(REV5_CATALOG_PATH)
rev4_docs = load_corpus_jsonl(REV4_CATALOG_PATH)
print(f"Rev5 docs: {len(rev5_docs)} | Rev4 docs: {len(rev4_docs)}")

print("Building BM25 for Rev.5 ...")
bm25_rev5, rev5_ids = build_bm25_from_corpus(rev5_docs, k1=BM25_K1, b=BM25_B)

print("Building BM25 for Rev.4 ...")
bm25_rev4, rev4_ids = build_bm25_from_corpus(rev4_docs, k1=BM25_K1, b=BM25_B)

print("\nEvaluating Rev.5 ...")
rev5_results = evaluate_gold(rev5_gold, bm25_rev5, rev5_ids, top_n=TOP_N)
rev5_results.to_csv(REV5_OUT, index=False)
print(f"Saved Rev.5 detailed results -> {REV5_OUT}")

print("\nEvaluating Rev.4 ...")
rev4_results = evaluate_gold(rev4_gold, bm25_rev4, rev4_ids, top_n=TOP_N)
rev4_results.to_csv(REV4_OUT, index=False)
print(f"Saved Rev.4 detailed results -> {REV4_OUT}")

# Print inline summary
display_summary_report(rev5_results, "NIST SP 800-53 Rev. 5")
display_summary_report(rev4_results, "NIST SP 800-53 Rev. 4")

# Save Markdown report
save_summary_report(rev5_results, rev4_results, out_path=SUMMARY_MD, system_name="System 1: BM25 Retriever")


Loading corpora ......
Rev5 docs: 1193 | Rev4 docs: 922
Building BM25 for Rev.5 ...
Building BM25 for Rev.4 ...

Evaluating Rev.5 ...
Saved Rev.5 detailed results -> S1_bm25_rev5_results.csv

Evaluating Rev.4 ...
Saved Rev.4 detailed results -> S1_bm25_rev4_results.csv

--- NIST SP 800-53 Rev. 5: Performance ---
| Metric    | Overall (100q) | ODP-Subset (63q) |
|-----------|------------------|--------------------|
| Recall@1  | 0.7600           | 0.8095             |
| Recall@5  | 0.9700           | 0.9683             |
| Recall@10 | 0.9900           | 0.9841             |
| MRR@10    | 0.8512           | 0.8907             |
| nDCG@10   | 0.8857           | 0.9144             |

--- NIST SP 800-53 Rev. 4: Performance ---
| Metric    | Overall (36q) | ODP-Subset (19q) |
|-----------|------------------|--------------------|
| Recall@1  | 0.6111           | 0.7368             |
| Recall@5  | 0.8611           | 0.8947             |
| Recall@10 | 0.9167           | 0.9474             |
| M

# Error Bank Re-evaluation

This section is for re-evaluating a "gold set" of questions where the model previously failed. This is useful for analyzing failure patterns or testing improvements.

**To use this section:**
1.  **Run All Cells Above:** Ensure the main evaluation has completed and the `bm25_rev5` and `bm25_rev4` models are in memory.
2.  **Prepare File:** Create a CSV file (e.g., `error_bank_v1.csv`) containing the questions needed to re-test.
3.  **Add `version` Column:** CSV **must** have a column named `version` that labels each question as `rev5` or `rev4` so it can be tested against the correct corpus.
4.  **Upload:** Upload error bank file to the Colab environment (the `/content/` directory).
5.  **Run:** Run the code cell below.

In [11]:
# === CONFIG ===

# add version column to CSV
VERSION_COLUMN_NAME = "version"

ERROR_BANK_REV5_OUT = "S1_bm25_error_bank_rev5_results.csv"
ERROR_BANK_REV4_OUT = "S1_bm25_error_bank_rev4_results.csv"

# --- Check if the error bank file exists before trying to run ---
if not os.path.exists(ERROR_BANK_CSV):
    print(f"\n--- Skipping Error Bank Evaluation ---")
    print(f"File not found: '{ERROR_BANK_CSV}'")
    print("To run this analysis, upload the error bank CSV and re-run this cell.")

else:
    print(f"\n--- Starting Split Error Bank Evaluation ---")
    print(f"Loading error bank: {ERROR_BANK_CSV} ...")

    try:
        # 1. Load the new CSV
        error_bank_gold = pd.read_csv(ERROR_BANK_CSV, encoding="ISO-8859-1")

        # 2. Check for the required 'version' column
        if VERSION_COLUMN_NAME not in error_bank_gold.columns:
            print(f"CRITICAL ERROR: CSV file '{ERROR_BANK_CSV}' is missing")
            print(f"a column named '{VERSION_COLUMN_NAME}'.")
            print("Please add this column and label each row as 'rev5' or 'rev4'.")

        else:
            # 3. Apply the same data cleaning
            error_bank_gold.dropna(subset=["question"], inplace=True)
            error_bank_gold["question"] = error_bank_gold["question"].astype(str).str.strip().str.lstrip(": ")
            error_bank_gold[VERSION_COLUMN_NAME] = error_bank_gold[VERSION_COLUMN_NAME].astype(str).str.lower().str.strip()
            print(f"Loaded {len(error_bank_gold)} total questions from error bank.")

            # 4. Split the DataFrame based on the 'version' column
            rev5_error_gold = error_bank_gold[error_bank_gold[VERSION_COLUMN_NAME] == 'rev5'].copy()
            rev4_error_gold = error_bank_gold[error_bank_gold[VERSION_COLUMN_NAME] == 'rev4'].copy()

            print(f"Found {len(rev5_error_gold)} Rev. 5 questions.")
            print(f"Found {len(rev4_error_gold)} Rev. 4 questions.")

            # --- 5. Evaluate Rev. 5 Error Set ---
            if not rev5_error_gold.empty:
                print("\nEvaluating Rev.5 error bank against Rev.5 corpus...")
                rev5_error_results = evaluate_gold(rev5_error_gold, bm25_rev5, rev5_ids, top_n=TOP_N)
                rev5_error_results.to_csv(ERROR_BANK_REV5_OUT, index=False)
                print(f"Saved Rev.5 error bank detailed results -> {ERROR_BANK_REV5_OUT}")

                # Display summary
                print(f"\n### Error Bank — Rev.5 only (n={len(rev5_error_gold)})")
                display_summary_report(rev5_error_results, "Error Bank (vs Rev. 5)")
            else:
                print("\nNo Rev. 5 questions found in error bank.")

            # --- 6. Evaluate Rev. 4 Error Set ---
            if not rev4_error_gold.empty:
                print("\nEvaluating Rev.4 error bank against Rev.4 corpus...")
                rev4_error_results = evaluate_gold(rev4_error_gold, bm25_rev4, rev4_ids, top_n=TOP_N)
                rev4_error_results.to_csv(ERROR_BANK_REV4_OUT, index=False)
                print(f"Saved Rev.4 error bank detailed results -> {ERROR_BANK_REV4_OUT}")

                # Display summary
                print(f"\n### Error Bank — Rev.4 only (n={len(rev4_error_gold)})")
                display_summary_report(rev4_error_results, "Error Bank (vs Rev. 4)")
            else:
                print("\nNo Rev. 4 questions found in error bank.")

        save_summary_report(
            rev5_results,
            rev4_results,
            out_path=SUMMARY_MD,
            system_name="System 1: BM25 Retriever",
            err_rev5_df=rev5_error_results,
            err_rev4_df=rev4_error_results
            )


    except KeyError as e:
        print(f"ERROR: A required column is missing: {e}. Did you add the '{VERSION_COLUMN_NAME}' column?")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")


--- Starting Split Error Bank Evaluation ---
Loading error bank: /content/drive/MyDrive/compliance_data/error_bank/error_bank_v1.csv ...
Loaded 34 total questions from error bank.
Found 23 Rev. 5 questions.
Found 11 Rev. 4 questions.

Evaluating Rev.5 error bank against Rev.5 corpus...
Saved Rev.5 error bank detailed results -> S1_bm25_error_bank_rev5_results.csv

### Error Bank — Rev.5 only (n=23)

--- Error Bank (vs Rev. 5): Performance ---
| Metric    | Overall (23q) | ODP-Subset (0q) |
|-----------|------------------|--------------------|
| Recall@1  | 0.0000           | 0.0000             |
| Recall@5  | 0.9130           | 0.0000             |
| Recall@10 | 1.0000           | 0.0000             |
| MRR@10    | 0.3966           | 0.0000             |
| nDCG@10   | 0.5465           | 0.0000             |

Evaluating Rev.4 error bank against Rev.4 corpus...
Saved Rev.4 error bank detailed results -> S1_bm25_error_bank_rev4_results.csv

### Error Bank — Rev.4 only (n=11)

--- Error B

Save outputs into Google Drive:

In [12]:
import os
import shutil

# --- 1. Define Destination ---
TARGET_DIR = "/content/drive/MyDrive/ablation_outputs/system_1_bm25"

# --- 2. Create Directory (if it doesn't exist) ---
os.makedirs(TARGET_DIR, exist_ok=True)
print(f"Ensured output directory exists: {TARGET_DIR}")

# --- 3. List of Files to Move ---
# (REV5_OUT, REV4_OUT, SUMMARY_MD, ERROR_BANK_REV5_OUT, ERROR_BANK_REV4_OUT)
files_to_move = [
    "S1_bm25_rev5_results.csv",
    "S1_bm25_rev4_results.csv",
    "S1_bm25_report.md",
    "S1_bm25_error_bank_rev5_results.csv",
    "S1_bm25_error_bank_rev4_results.csv"
]

# --- 4. Move Files ---
print("\nMoving generated files to Google Drive...")
moved_files_count = 0
skipped_files_count = 0

for file_name in files_to_move:
    # Source path is the temporary Colab /content/ folder
    source_path = os.path.join("/content/", file_name)

    # Destination path
    dest_path = os.path.join(TARGET_DIR, file_name)

    # Check if the file exists in /content/ before moving
    if os.path.exists(source_path):
        try:
            shutil.move(source_path, dest_path)
            print(f"Successfully moved: {file_name}")
            moved_files_count += 1
        except Exception as e:
            print(f"ERROR moving {file_name}: {e}")
    else:
        # This will happen if, for example, the error bank wasn't run
        print(f"  - Skipping (not found): {file_name}")
        skipped_files_count += 1

print(f"\nMove complete: {moved_files_count} files moved, {skipped_files_count} files skipped.")

Ensured output directory exists: /content/drive/MyDrive/ablation_outputs/system_1_bm25

Moving generated files to Google Drive...
Successfully moved: S1_bm25_rev5_results.csv
Successfully moved: S1_bm25_rev4_results.csv
Successfully moved: S1_bm25_report.md
Successfully moved: S1_bm25_error_bank_rev5_results.csv
Successfully moved: S1_bm25_error_bank_rev4_results.csv

Move complete: 5 files moved, 0 files skipped.


# README: System 2 — Dense Retriever (e5-small-v2)

 **System 2 (Dense Retriever)**, the second phase of the ComplianceGPT retrieval experiments. This system replaces the lexical (keyword-based) BM25 retriever from System 1 with a modern, semantic (meaning-based) dense retriever.

In [13]:
# === SYSTEM 2: DENSE RETRIEVER ===
# --- 1. S2 Dependencies ---

!pip install -q sentence-transformers faiss-cpu

import torch
import faiss
from sentence_transformers import SentenceTransformer
import time

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 76.0 MB/s eta 0:00:00


In [14]:
# --- 2. S2 Helper Functions & Classes
import pandas as pd
from typing import List, Dict, Tuple, Any, Optional
import time
import numpy as np

def evaluate_gold_modular(gold_df: pd.DataFrame, retriever: Any, top_n=10) -> pd.DataFrame:
    """
    Runs evaluation on a gold set DataFrame using a provided retriever object.
    """
    records = []
    start_time = time.time()
    total_queries = len(gold_df)

    for i, row in gold_df.iterrows():
        q = str(row.get("question", "")).strip()
        gold_cid = str(row.get("control_id", "")).strip().upper()
        odp = str(row.get("odp_required", ""))

        if not q or not gold_cid: continue

        try:
            retrieved_docs_scores = retriever.search(q, top_k=top_n)
        except Exception as e:
            print(f"Error during search for query '{q[:50]}...': {e}")
            retrieved_docs_scores = []

        retrieved_ids = [str(doc_id).upper() for doc_id, score in retrieved_docs_scores]
        rank = 0
        if gold_cid in retrieved_ids:
            rank = retrieved_ids.index(gold_cid) + 1 # 1-based index

        top_1_hit = retrieved_ids[0] if retrieved_ids else ""

        records.append({
            "question": q, "control_id": gold_cid, "rank": rank,
            "top_1_hit": top_1_hit, "is_hit_at_1": rank == 1,
            "is_hit_at_5": 1 <= rank <= 5, "is_hit_at_10": 1 <= rank <= top_n,
            "retrieved_ids": retrieved_ids, "odp_required": odp
        })

        if (i + 1) % 25 == 0 or (i + 1) == total_queries:
            print(f"  Processed {i+1}/{total_queries} queries...")

    end_time = time.time()
    avg_time = (end_time - start_time) / total_queries if total_queries > 0 else 0
    print(f"Evaluation complete. Average query time: {avg_time:.4f} seconds")

    return pd.DataFrame(records)


# 1. We remove 'control_id' (it pollutes semantic meaning for e5).
# 2. We correctly read the NESTED 'control_text' and 'control_guidance'.
def extract_text_from_record_S2(record: Dict[str, Any]) -> str:
    """Extracts text for DENSE encoding.
       - *INCLUDES* the control_id.
       - *CORRECTLY* reads the nested text.
       - This is now an IDENTICAL copy of the S1 'extract_text_from_record'.
    """
    text_parts = []
    text_parts.append(record.get('control_id', ''))
    text_parts.append(record.get('control_name', ''))

    # read nested text
    control_text = record.get('control_text', {}).get('text', '')
    if control_text:
        text_parts.append(control_text)

    # read nested text
    guidance = record.get('control_guidance', {}).get('text', '')
    if guidance:
        text_parts.append("Guidance:")
        text_parts.append(guidance)

    return " ".join(filter(None, text_parts)).strip()

# --- Dense Index (uses RAW text) ---
class DenseIndex:
    def __init__(self, model_id: str, batch_size: int = 128):
        self.model_id = model_id
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"Initializing SentenceTransformer on device: {self.device}")
        self.model = SentenceTransformer(model_id, device=self.device)
        self.batch_size = batch_size
        self.index = None
        self.ids: List[str] = []
        self.dim = self.model.get_sentence_embedding_dimension()
        print(f"Model embedding dimension: {self.dim}")

    def encode_passages(self, texts: List[str]) -> np.ndarray:
        print(f"Encoding {len(texts)} passages...")
        # e5 model expects the "passage: " prefix
        X = self.model.encode([f"passage: {t}" for t in texts],
                              batch_size=self.batch_size,
                              convert_to_numpy=True,
                              normalize_embeddings=True,
                              show_progress_bar=True)
        return X.astype('float32')

    def encode_queries(self, qs: List[str]) -> np.ndarray:
        # e5 model expects the "query: " prefix
        Q = self.model.encode([f"query: {q}" for q in qs],
                              batch_size=self.batch_size,
                              convert_to_numpy=True,
                              normalize_embeddings=True,
                              show_progress_bar=False)
        return Q.astype('float32')

    def build(self, texts: List[str], ids: List[str]):
        self.ids = ids[:]
        # Pass the RAW texts (from the S1-style extractor) to the encoder
        X = self.encode_passages(texts)
        self.index = faiss.IndexFlatIP(self.dim)
        self.index.add(X)
        print(f"FAISS index built: {self.index.ntotal} vectors")

    def search(self, query: str, top_k=50) -> List[Tuple[str, float]]:
        if not self.index:
            raise RuntimeError("Index has not been built. Call .build() first.")
        # Pass the RAW query to the encoder
        qv = self.encode_queries([query])
        scores, idxs = self.index.search(qv, top_k)
        idxs_list = idxs[0].tolist()
        scores_list = scores[0].tolist()
        return [(self.ids[i], float(s)) for i, s in zip(idxs_list, scores_list) if i != -1]

class DenseRetriever:
    def __init__(self, dense_index: DenseIndex, top_k=50):
        self.dense_index = dense_index
        self.top_k = top_k

    def search(self, query: str, top_k: Optional[int]=None) -> List[Tuple[str, float]]:
        k = top_k or self.top_k
        return self.dense_index.search(query, top_k=k)

In [15]:
# --- 3. S2 Configuration ---

DENSE_MODEL_ID = 'intfloat/e5-small-v2'

# System 2 Output Paths
S2_OUTPUT_DIR = "/content/drive/MyDrive/ablation_outputs/system_2_dense"
S2_REV5_OUT = "S2_dense_rev5_results.csv"
S2_REV4_OUT = "S2_dense_rev4_results.csv"
S2_SUMMARY_MD = os.path.join(S2_OUTPUT_DIR, "S2_dense_report.md")
S2_ERROR_BANK_REV5_OUT = "S2_dense_error_bank_rev5_results.csv"
S2_ERROR_BANK_REV4_OUT = "S2_dense_error_bank_rev4_results.csv"

In [16]:
# ===================================================================
# --- 4. S2 MAIN: Build, Load, and Evaluate All ---
# ===================================================================

# --- 1. Create Output Directory ---
os.makedirs(S2_OUTPUT_DIR, exist_ok=True)
print(f"S2 Output directory: {S2_OUTPUT_DIR}")

# --- 2. Build Rev. 5 Dense Index ---
print("\n--- S2: Building Rev. 5 Index ---")
rev5_corpus_s2 = load_corpus_jsonl(REV5_CATALOG_PATH)
rev5_ids_s2 = [str(r.get('control_id', '')).upper() for r in rev5_corpus_s2]
rev5_docs_s2 = [r.get('text', '') for r in rev5_corpus_s2] # <--- THIS IS THE FIX
dense_index_rev5 = DenseIndex(model_id=DENSE_MODEL_ID)
dense_index_rev5.build(rev5_docs_s2, rev5_ids_s2)
dense_retriever_rev5 = DenseRetriever(dense_index_rev5, top_k=TOP_N)
print("S2: Rev. 5 Dense Retriever is ready.")

# --- 3. Build Rev. 4 Dense Index ---
print("\n--- S2: Building Rev. 4 Index ---")
rev4_corpus_s2 = load_corpus_jsonl(REV4_CATALOG_PATH)
rev4_ids_s2 = [str(r.get('control_id', '')).upper() for r in rev4_corpus_s2]
rev4_docs_s2 = [r.get('text', '') for r in rev4_corpus_s2]
dense_index_rev4 = DenseIndex(model_id=DENSE_MODEL_ID)
dense_index_rev4.build(rev4_docs_s2, rev4_ids_s2)
dense_retriever_rev4 = DenseRetriever(dense_index_rev4, top_k=TOP_N)
print("S2: Rev. 4 Dense Retriever is ready.")

# --- 4. Load All Gold Sets ---
# (This assumes REV5_GOLD_CSV, REV4_GOLD_CSV, and ERROR_BANK_CSV are
print("\n--- S2: Loading Gold Sets ---")
s2_rev5_results = None
s2_rev4_results = None
s2_err_rev5_results = None
s2_err_rev4_results = None
rev5_error_gold = pd.DataFrame()
rev4_error_gold = pd.DataFrame()

try:
    rev5_gold = pd.read_csv(REV5_GOLD_CSV, encoding="ISO-8859-1")
    rev5_gold.dropna(subset=["question"], inplace=True)
    rev5_gold["question"] = rev5_gold["question"].astype(str).str.strip().str.lstrip(": ")
    print(f"Loaded {len(rev5_gold)} Rev. 5 gold queries.")

    rev4_gold = pd.read_csv(REV4_GOLD_CSV, encoding="ISO-8859-1")
    rev4_gold.dropna(subset=["question"], inplace=True)
    rev4_gold["question"] = rev4_gold["question"].astype(str).str.strip().str.lstrip(": ")
    print(f"Loaded {len(rev4_gold)} Rev. 4 gold queries.")

    VERSION_COLUMN_NAME = "version"
    error_bank_gold = pd.read_csv(ERROR_BANK_CSV, encoding="ISO-8859-1")
    error_bank_gold.dropna(subset=["question"], inplace=True)
    error_bank_gold["question"] = error_bank_gold["question"].astype(str).str.strip().str.lstrip(": ")
    error_bank_gold[VERSION_COLUMN_NAME] = error_bank_gold[VERSION_COLUMN_NAME].astype(str).str.lower().str.strip()
    print(f"Loaded {len(error_bank_gold)} total error bank queries.")

    rev5_error_gold = error_bank_gold[error_bank_gold[VERSION_COLUMN_NAME] == 'rev5'].copy()
    rev4_error_gold = error_bank_gold[error_bank_gold[VERSION_COLUMN_NAME] == 'rev4'].copy()
    print(f"Found {len(rev5_error_gold)} Rev. 5 error questions.")
    print(f"Found {len(rev4_error_gold)} Rev. 4 error questions.")

except FileNotFoundError as e: print(f"\nCRITICAL ERROR: Could not find file. {e}")
except NameError as e:
    print(f"\nCRITICAL ERROR: A variable from S1 config is missing. {e}")
    print("Please ensure 'REV5_CATALOG_PATH', 'REV4_CATALOG_PATH', 'REV5_GOLD_CSV', 'REV4_GOLD_CSV', and 'ERROR_BANK_CSV' are all defined in S1 config cell.")
except Exception as e: print(f"Error loading gold sets: {e}")

# --- 5. Run All S2 Evaluations ---
if 'rev5_gold' in locals():
    print("\n--- Running S2 Rev. 5 Evaluation ---")
    s2_rev5_results = evaluate_gold_modular(rev5_gold, dense_retriever_rev5, top_n=TOP_N)
    s2_rev5_results.to_csv(os.path.join(S2_OUTPUT_DIR, S2_REV5_OUT), index=False)
    print(f"Saved S2 Rev. 5 detailed results -> {S2_REV5_OUT}")

if 'rev4_gold' in locals():
    print("\n--- Running S2 Rev. 4 Evaluation ---")
    s2_rev4_results = evaluate_gold_modular(rev4_gold, dense_retriever_rev4, top_n=TOP_N)
    s2_rev4_results.to_csv(os.path.join(S2_OUTPUT_DIR, S2_REV4_OUT), index=False)
    print(f"Saved S2 Rev. 4 detailed results -> {S2_REV4_OUT}")

if not rev5_error_gold.empty:
    print("\n--- Running S2 Error Bank (Rev. 5) Evaluation ---")
    s2_err_rev5_results = evaluate_gold_modular(rev5_error_gold, dense_retriever_rev5, top_n=TOP_N)
    s2_err_rev5_results.to_csv(os.path.join(S2_OUTPUT_DIR, S2_ERROR_BANK_REV5_OUT), index=False)
    print(f"Saved S2 Rev. 5 error bank detailed results -> {S2_ERROR_BANK_REV5_OUT}")

if not rev4_error_gold.empty:
    print("\n--- Running S2 Error Bank (Rev. 4) Evaluation ---")
    s2_err_rev4_results = evaluate_gold_modular(rev4_error_gold, dense_retriever_rev4, top_n=TOP_N)
    s2_err_rev4_results.to_csv(os.path.join(S2_OUTPUT_DIR, S2_ERROR_BANK_REV4_OUT), index=False)
    print(f"Saved S2 Rev. 4 error bank detailed results -> {S2_ERROR_BANK_REV4_OUT}")

# --- 6. Display S2 Summary Report ---
print("\n\n--- FINAL S2 SUMMARY REPORTS ---")

# --- Main Report ---
if s2_rev5_results is not None and s2_rev4_results is not None:
    display_summary_report(s2_rev5_results, "S2 Dense - NIST SP 800-53 Rev. 5")
    display_summary_report(s2_rev4_results, "S2 Dense - NIST SP 800-53 Rev. 4")

    # --- Error Bank Reports (if they exist) ---
    if s2_err_rev5_results is not None:
        print(f"\n### S2 Error Bank — Rev.5 only (n={len(rev5_error_gold)})")
        display_summary_report(s2_err_rev5_results, "S2 Dense - Error Bank (vs Rev. 5)")

    if s2_err_rev4_results is not None:
        print(f"\n### S2 Error Bank — Rev.4 only (n={len(rev4_error_gold)})")
        display_summary_report(s2_err_rev4_results, "S2 Dense - Error Bank (vs Rev. 4)")

    # --- Save the complete .md report ---
    print(f"\nSaving summary report to {S2_SUMMARY_MD}...")

    save_summary_report(
            s2_rev5_results,
            s2_rev4_results,
            out_path=S2_SUMMARY_MD,
            system_name="S2 Dense Retriever",
            err_rev5_df=s2_err_rev5_results,
            err_rev4_df=s2_err_rev4_results
            )

else:
    print("Could not generate main summary report. S2 Rev 5 or Rev 4 results are missing.")

S2 Output directory: /content/drive/MyDrive/ablation_outputs/system_2_dense

--- S2: Building Rev. 5 Index ---
Initializing SentenceTransformer on device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

Model embedding dimension: 384
Encoding 1193 passages...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

FAISS index built: 1193 vectors
S2: Rev. 5 Dense Retriever is ready.

--- S2: Building Rev. 4 Index ---
Initializing SentenceTransformer on device: cuda
Model embedding dimension: 384
Encoding 922 passages...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

FAISS index built: 922 vectors
S2: Rev. 4 Dense Retriever is ready.

--- S2: Loading Gold Sets ---
Loaded 100 Rev. 5 gold queries.
Loaded 36 Rev. 4 gold queries.
Loaded 34 total error bank queries.
Found 23 Rev. 5 error questions.
Found 11 Rev. 4 error questions.

--- Running S2 Rev. 5 Evaluation ---
  Processed 25/100 queries...
  Processed 50/100 queries...
  Processed 75/100 queries...
  Processed 100/100 queries...
Evaluation complete. Average query time: 0.0111 seconds
Saved S2 Rev. 5 detailed results -> S2_dense_rev5_results.csv

--- Running S2 Rev. 4 Evaluation ---
  Processed 25/36 queries...
  Processed 36/36 queries...
Evaluation complete. Average query time: 0.0104 seconds
Saved S2 Rev. 4 detailed results -> S2_dense_rev4_results.csv

--- Running S2 Error Bank (Rev. 5) Evaluation ---
  Processed 23/23 queries...
Evaluation complete. Average query time: 0.0103 seconds
Saved S2 Rev. 5 error bank detailed results -> S2_dense_error_bank_rev5_results.csv

--- Running S2 Error Ban

# System 3: BM25 + Query Rewrites (S1 BM25 + S3 Queries)

This section implements **System 3** of the experiment.

It re-uses the **BM25 models (`bm25_rev5`, `bm25_rev4`)** built for System 1, but evaluates them against the new **Query Rewrite (QUR)** gold sets.

**Prerequisite:** All System 1 cells must be run first to load the BM25 models and evaluation functions into memory.

In [17]:
# ===================================================================
# --- S3 & S4: CONFIGURATION ---
# ===================================================================

print("--- Initializing S3 & S4 Configuration ---")

# --- 1. S3 (Rewrite-Only) Output Paths ---
S3_OUTPUT_DIR = "/content/drive/MyDrive/ablation_outputs/system_3_rewrite_only"
os.makedirs(S3_OUTPUT_DIR, exist_ok=True)
print(f"S3 Output directory: {S3_OUTPUT_DIR}")

S3_REV5_OUT = os.path.join(S3_OUTPUT_DIR, "S3_qur_replace_rev5_results.csv")
S3_REV4_OUT = os.path.join(S3_OUTPUT_DIR, "S3_qur_replace_rev4_results.csv")
S3_ERR_REV5_OUT = os.path.join(S3_OUTPUT_DIR, "S3_qur_replace_error_bank_rev5_results.csv")
S3_ERR_REV4_OUT = os.path.join(S3_OUTPUT_DIR, "S3_qur_replace_error_bank_rev4_results.csv")
S3_SUMMARY_MD = os.path.join(S3_OUTPUT_DIR, "S3_qur_replace_report.md")

# --- 2. S4 (QUR+RRF) Output Paths ---
S4_OUTPUT_DIR = "/content/drive/MyDrive/ablation_outputs/system_4_qur_rrf"
os.makedirs(S4_OUTPUT_DIR, exist_ok=True)
print(f"S4 Output directory: {S4_OUTPUT_DIR}")

S4_REV5_OUT = os.path.join(S4_OUTPUT_DIR, "S4_qur_rrf_rev5_results.csv")
S4_REV4_OUT = os.path.join(S4_OUTPUT_DIR, "S4_qur_rrf_rev4_results.csv")
S4_ERR_REV5_OUT = os.path.join(S4_OUTPUT_DIR, "S4_qur_rrf_error_bank_rev5_results.csv")
S4_ERR_REV4_OUT = os.path.join(S4_OUTPUT_DIR, "S4_qur_rrf_error_bank_rev4_results.csv")
S4_SUMMARY_MD = os.path.join(S4_OUTPUT_DIR, "S4_qur_rrf_report.md")

# --- 3. Input QUR CSV Paths ---
QUR_CSV_REV5 = "/content/drive/MyDrive/compliance_outputs/outputs_qur/qur_rewrites_rev5.csv"
QUR_CSV_REV4 = "/content/drive/MyDrive/compliance_outputs/outputs_qur/qur_rewrites_rev4.csv"
QUR_CSV_ERRB = "/content/drive/MyDrive/compliance_outputs/outputs_qur/qur_rewrites_error_bank.csv"

--- Initializing S3 & S4 Configuration ---
S3 Output directory: /content/drive/MyDrive/ablation_outputs/system_3_rewrite_only
S4 Output directory: /content/drive/MyDrive/ablation_outputs/system_4_qur_rrf


In [18]:
# ===================================================================
# --- SYSTEM 3: BM25 (REWRITE-ONLY) ABLATION ---
# ===================================================================

def prepare_s3_rewrite_only_set(s1_gold_df, s3_qur_df):
    """
    Prepares the 'Rewrite-Only' gold set by merging the S1 gold set
    with ONLY the single best rewrite (rank 1).
    This function REPLACES the original query.
    """
    # --- Clean columns BEFORE merging ---
    s1_gold_df_cleaned = s1_gold_df.copy()
    s1_gold_df_cleaned["question"] = s1_gold_df_cleaned["question"].astype(str).str.strip().str.lstrip(": ")

    # Filter for the best rewrite (rank 1) and clean 'original_query'
    s3_best_qur = s3_qur_df[s3_qur_df['rewrite_rank'] == 1].copy()
    s3_best_qur["original_query"] = s3_best_qur["original_query"].astype(str).str.strip().str.lstrip(": ")

    # Select only the needed columns from the rewrite file
    s3_best_qur_cols = s3_best_qur[['original_query', 'rewritten_query']]

    # --- Merge on text ---
    merged_df = pd.merge(
        s1_gold_df_cleaned,
        s3_best_qur_cols,
        left_on='question',
        right_on='original_query',
        how='inner'
    )

    if len(merged_df) == 0:
        print(f"WARNING: S3 Merge resulted in 0 rows. Check 'question' (S1) and 'original_query' (S3) columns.")
        return pd.DataFrame(columns=['question', 'control_id', 'odp_required', 'version'])

    # Drop the original 'question' and 'original_query'
    merged_df = merged_df.drop(columns=['question', 'original_query'])

    # Rename 'rewritten_query' to 'question' for the S1 evaluator
    merged_df = merged_df.rename(columns={'rewritten_query': 'question'})

    # Clean the new question column
    merged_df["question"] = merged_df["question"].astype(str).str.strip().str.lstrip(": ")

    # Ensure all required columns exist
    if 'odp_required' not in merged_df.columns:
        merged_df['odp_required'] = 'N/A'
    if 'version' not in merged_df.columns:
        merged_df['version'] = 'N/A' # Will be overwritten for error bank

    return merged_df

In [19]:
# --- S3 (Rewrite-Only) MAIN Evaluation ---
print("--- Starting System 3 (Rewrite-Only) Evaluation ---")

try:
    # --- 1. Load S1 Ground Truth Sets ---
    s1_rev5_gold = pd.read_csv(REV5_GOLD_CSV, encoding="ISO-8859-1")
    s1_rev4_gold = pd.read_csv(REV4_GOLD_CSV, encoding="ISO-8859-1")
    s1_error_bank = pd.read_csv(ERROR_BANK_CSV, encoding="ISO-8859-1")

    # --- 2. Load S3 Query Rewrite Sets ---
    s3_rev5_qur = pd.read_csv(QUR_CSV_REV5, encoding="ISO-8859-1")
    s3_rev4_qur = pd.read_csv(QUR_CSV_REV4, encoding="ISO-8859-1")
    s3_errb_qur = pd.read_csv(QUR_CSV_ERRB, encoding="ISO-8859-1")

    # --- 3. Evaluate S3 Rev 5 (Rewrite-Only) ---
    print("\n--- Running S3 (Rewrite-Only) Rev. 5 Evaluation ---")
    s3_rev5_gold = prepare_s3_rewrite_only_set(s1_rev5_gold, s3_rev5_qur)
    print(f"Prepared {len(s3_rev5_gold)} merged S3 Rev. 5 queries.")

    s3_rev5_results = pd.DataFrame()
    if not s3_rev5_gold.empty:
        # Use the S1 evaluator (evaluate_gold)
        s3_rev5_results = evaluate_gold(s3_rev5_gold, bm25_rev5, rev5_ids, top_n=TOP_N)
        s3_rev5_results.to_csv(S3_REV5_OUT, index=False)
        print(f"Saved S3 Rev. 5 detailed results -> {S3_REV5_OUT}")
        display_summary_report(s3_rev5_results, "S3 (Rewrite-Only) - NIST SP 800-53 Rev. 5")

    # --- 4. Evaluate S3 Rev 4 (Rewrite-Only) ---
    print("\n--- Running S3 (Rewrite-Only) Rev. 4 Evaluation ---")
    s3_rev4_gold = prepare_s3_rewrite_only_set(s1_rev4_gold, s3_rev4_qur)
    print(f"Prepared {len(s3_rev4_gold)} merged S3 Rev. 4 queries.")

    s3_rev4_results = pd.DataFrame()
    if not s3_rev4_gold.empty:
        s3_rev4_results = evaluate_gold(s3_rev4_gold, bm25_rev4, rev4_ids, top_n=TOP_N)
        s3_rev4_results.to_csv(S3_REV4_OUT, index=False)
        print(f"Saved S3 Rev. 4 detailed results -> {S3_REV4_OUT}")
        display_summary_report(s3_rev4_results, "S3 (Rewrite-Only) - NIST SP 800-53 Rev. 4")

    # --- 5. Evaluate S3 Error Bank (Rewrite-Only) ---
    print("\n--- Running S3 (Rewrite-Only) Error Bank Evaluation ---")
    s3_error_bank_gold = prepare_s3_rewrite_only_set(s1_error_bank, s3_errb_qur)
    print(f"Prepared {len(s3_error_bank_gold)} total merged S3 error bank queries.")

    s3_err_rev5_results = pd.DataFrame()
    s3_err_rev4_results = pd.DataFrame()
    if not s3_error_bank_gold.empty:
        s3_error_bank_gold['version'] = s3_error_bank_gold['version'].astype(str).str.lower().str.strip()
        s3_err_rev5_gold = s3_error_bank_gold[s3_error_bank_gold['version'] == 'rev5'].copy()
        s3_err_rev4_gold = s3_error_bank_gold[s3_error_bank_gold['version'] == 'rev4'].copy()

        if not s3_err_rev5_gold.empty:
            print(f"\nEvaluating {len(s3_err_rev5_gold)} S3 Rev. 5 error bank queries...")
            s3_err_rev5_results = evaluate_gold(s3_err_rev5_gold, bm25_rev5, rev5_ids, top_n=TOP_N)
            s3_err_rev5_results.to_csv(S3_ERR_REV5_OUT, index=False)
            display_summary_report(s3_err_rev5_results, "S3 (Rewrite-Only) Error Bank (vs Rev. 5)")

        if not s3_err_rev4_gold.empty:
            print(f"\nEvaluating {len(s3_err_rev4_gold)} S3 Rev. 4 error bank queries...")
            s3_err_rev4_results = evaluate_gold(s3_err_rev4_gold, bm25_rev4, rev4_ids, top_n=TOP_N)
            s3_err_rev4_results.to_csv(S3_ERR_REV4_OUT, index=False)
            display_summary_report(s3_err_rev4_results, "S3 (Rewrite-Only) Error Bank (vs Rev. 4)")

    # --- 6. Save Final S3 Report ---
    if not s3_rev5_results.empty and not s3_rev4_results.empty:
        print(f"\nSaving S3 summary report to {S3_SUMMARY_MD}...")
        save_summary_report(
            s3_rev5_results, s3_rev4_results,
            out_path=S3_SUMMARY_MD,
            system_name="System 3: BM25 (Rewrite-Only)",
            err_rev5_df=s3_err_rev5_results,
            err_rev4_df=s3_err_rev4_results
        )
    else:
        print("\nCould not save S3 summary report.")

except Exception as e:
    print(f"\nAn unexpected error occurred in S3: {e}")

--- Starting System 3 (Rewrite-Only) Evaluation ---

--- Running S3 (Rewrite-Only) Rev. 5 Evaluation ---
Prepared 100 merged S3 Rev. 5 queries.
Saved S3 Rev. 5 detailed results -> /content/drive/MyDrive/ablation_outputs/system_3_rewrite_only/S3_qur_replace_rev5_results.csv

--- S3 (Rewrite-Only) - NIST SP 800-53 Rev. 5: Performance ---
| Metric    | Overall (100q) | ODP-Subset (63q) |
|-----------|------------------|--------------------|
| Recall@1  | 0.5000           | 0.5238             |
| Recall@5  | 0.7400           | 0.7619             |
| Recall@10 | 0.8500           | 0.8730             |
| MRR@10    | 0.6085           | 0.6400             |
| nDCG@10   | 0.6660           | 0.6959             |

--- Running S3 (Rewrite-Only) Rev. 4 Evaluation ---
Prepared 36 merged S3 Rev. 4 queries.
Saved S3 Rev. 4 detailed results -> /content/drive/MyDrive/ablation_outputs/system_3_rewrite_only/S3_qur_replace_rev4_results.csv

--- S3 (Rewrite-Only) - NIST SP 800-53 Rev. 4: Performance ---
| M

## - **S4 (QUR‑RRF):**
Proved that fusion (RRF) is the correct way to use query rewrites with BM25, creating a system that beats the S1 baseline on harder (Rev. 4) queries.  



In [20]:
# ===================================================================
# --- SYSTEM 4: BM25 (QUR+RRF) ---
# ===================================================================

# --- S4 Helper Function (RRF) ---

def get_rrf_fused_rank(
    original_query: str,
    gold_cid: str,
    bm25: BM25Okapi,
    control_ids: List[str],
    qur_df: pd.DataFrame,
    top_n: int = 10,
    rrf_k: int = 60
) -> int:
    """
    Performs a QUR evaluation for a single query using RRF.

    1. Finds all rewrites for the original_query in the qur_df.
    2. Runs BM25 for the original query AND all its rewrites.
    3. Fuses the results using Reciprocal Rank Fusion (RRF).
    4. Finds the final rank of the gold_cid in the fused list.
    """

    # --- 1. Get all queries (original + rewrites) ---

    # Clean the single string query from the S1 gold set
    q_cleaned = original_query.strip().lstrip(": ")

    # Find all rows matching the original query
    # (The .str accessor is correct here because it's on a pandas Series)
    matching_rows = qur_df[qur_df['original_query'] == q_cleaned]

    queries_to_run = [original_query] # Start with the original

    if not matching_rows.empty:
        # Add all rewrites
        # (The .str accessor is correct here too)
        queries_to_run.extend(matching_rows['rewritten_query'].astype(str).str.strip().str.lstrip(": ").tolist())

    # Remove duplicates just in case
    queries_to_run = list(dict.fromkeys(queries_to_run))


    # --- 2. Run BM25 for all queries ---
    all_results_lists = []
    for q in queries_to_run:
        q_norm = normalize_text(q)
        q_tokens = tokenize(q_norm)
        idxs = bm25.get_top_n(q_tokens, n=top_n)
        retrieved_cids = [control_ids[i] for i in idxs]
        all_results_lists.append(retrieved_cids)


    # --- 3. Apply Reciprocal Rank Fusion (RRF) ---
    rrf_scores = {}
    for result_list in all_results_lists:
        for rank, cid in enumerate(result_list, start=1):
            rrf_score = 1.0 / (rrf_k + rank)
            rrf_scores[cid] = rrf_scores.get(cid, 0.0) + rrf_score

    # --- 4. Get final fused ranking ---
    if not rrf_scores:
        return 0 # No results found at all

    # Sort by score, descending
    fused_ranking = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)
    fused_cids = [cid for cid, score in fused_ranking]

    # --- 5. Find rank of the gold_cid ---
    try:
        # 1-based rank
        final_rank = fused_cids.index(gold_cid) + 1
        return final_rank
    except ValueError:
        return 0 # Not found in the fused top results

In [21]:
# ===================================================================
# --- S4 MAIN: Load S1 Gold Sets, S3 Rewrites, and Evaluate w/ RRF ---
# ===================================================================
# This re-uses bm25_rev5, bm25_rev4, rev5_ids, and rev4_ids from S1

print("--- S4 (QUR+RRF) Evaluation ---")
print("Loading S1 Ground Truth and S4 (QUR) Rewrites...")

try:
    # --- 1. Load S1 Ground Truth Sets ---
    s1_rev5_gold = pd.read_csv(REV5_GOLD_CSV, encoding="ISO-8859-1")
    s1_rev4_gold = pd.read_csv(REV4_GOLD_CSV, encoding="ISO-8859-1")
    s1_error_bank = pd.read_csv(ERROR_BANK_CSV, encoding="ISO-8859-1")

    # --- 2. Load QUR Rewrite Sets (we need all rewrites) ---
    s4_rev5_qur = pd.read_csv(QUR_CSV_REV5, encoding="ISO-8859-1")
    s4_rev4_qur = pd.read_csv(QUR_CSV_REV4, encoding="ISO-8859-1")
    s4_errb_qur = pd.read_csv(QUR_CSV_ERRB, encoding="ISO-8859-1")

    # Clean the join column in the rewrite files ONCE
    for df in [s4_rev5_qur, s4_rev4_qur, s4_errb_qur]:
        if 'original_query' in df.columns:
            df['original_query'] = df['original_query'].astype(str).str.strip().str.lstrip(": ")

    # --- 3. Evaluate S4 Rev 5 (QUR+RRF) ---
    print("\n--- Running S4 (QUR+RRF) Rev. 5 Evaluation ---")
    s4_rev5_records = []

    # Loop over the ORIGINAL S1 gold set
    for _, row in s1_rev5_gold.iterrows():
        q = row['question']
        cid = row['control_id']
        # (FIX) Safely get 'odp_required', default to "N/A"
        odp = str(row.get("odp_required", "N/A")).strip()

        final_rank = get_rrf_fused_rank(
            original_query=q,
            gold_cid=cid,
            bm25=bm25_rev5,
            control_ids=rev5_ids,
            qur_df=s4_rev5_qur,
            top_n=TOP_N
        )

        # (FIX) Append a new dictionary, not the 'row' Series
        s4_rev5_records.append({
            "question": q,
            "control_id": cid,
            "odp_required": odp,
            "rank": final_rank
        })

    s4_rev5_results = pd.DataFrame(s4_rev5_records)
    s4_rev5_results.to_csv(S4_REV5_OUT, index=False)
    print(f"Saved S4 Rev. 5 detailed results -> {S4_REV5_OUT}")
    display_summary_report(s4_rev5_results, "S4 (QUR+RRF) - NIST SP 800-53 Rev. 5")

    # --- 4. Evaluate S4 Rev 4 (QUR+RRF) ---
    print("\n--- Running S4 (QUR+RRF) Rev. 4 Evaluation ---")
    s4_rev4_records = []

    for _, row in s1_rev4_gold.iterrows():
        q = row['question']
        cid = row['control_id']
        # (FIX) Safely get 'odp_required', default to "N/A"
        odp = str(row.get("odp_required", "N/A")).strip()

        final_rank = get_rrf_fused_rank(
            original_query=q,
            gold_cid=cid,
            bm25=bm25_rev4,
            control_ids=rev4_ids,
            qur_df=s4_rev4_qur,
            top_n=TOP_N
        )

        # (FIX) Append a new dictionary, not the 'row' Series
        s4_rev4_records.append({
            "question": q,
            "control_id": cid,
            "odp_required": odp,
            "rank": final_rank
        })

    s4_rev4_results = pd.DataFrame(s4_rev4_records)
    s4_rev4_results.to_csv(S4_REV4_OUT, index=False)
    print(f"Saved S4 Rev. 4 detailed results -> {S4_REV4_OUT}")
    display_summary_report(s4_rev4_results, "S4 (QUR+RRF) - NIST SP 800-53 Rev. 4")

    # --- 5. Evaluate S4 Error Bank (QUR+RRF) ---
    print("\n--- Running S4 (QUR+RRF) Error Bank Evaluation ---")
    s4_err_rev5_records = []
    s4_err_rev4_records = []
    s4_err_rev5_results = pd.DataFrame() # Init empty
    s4_err_rev4_results = pd.DataFrame() # Init empty

    # Split the S1 error bank
    s1_error_bank['version'] = s1_error_bank['version'].astype(str).str.lower().str.strip()
    s1_err_rev5_gold = s1_error_bank[s1_error_bank['version'] == 'rev5']
    s1_err_rev4_gold = s1_error_bank[s1_error_bank['version'] == 'rev4']

    if not s1_err_rev5_gold.empty:
        print(f"\nEvaluating {len(s1_err_rev5_gold)} S4 Rev. 5 error bank queries...")
        for _, row in s1_err_rev5_gold.iterrows():
            q = row['question']
            cid = row['control_id']
            # (FIX) Safely get 'odp_required', default to "N/A"
            odp = str(row.get("odp_required", "N/A")).strip()

            final_rank = get_rrf_fused_rank(
                original_query=q,
                gold_cid=cid,
                bm25=bm25_rev5,
                control_ids=rev5_ids,
                qur_df=s4_errb_qur,
                top_n=TOP_N
            )

            # (FIX) Append a new dictionary, not the 'row' Series
            s4_err_rev5_records.append({
                "question": q,
                "control_id": cid,
                "odp_required": odp,
                "rank": final_rank,
                "version": "rev5" # Add version for the report
            })

        s4_err_rev5_results = pd.DataFrame(s4_err_rev5_records)

        # --- THIS IS THE FIX ---
        s4_err_rev5_results.to_csv(S4_ERR_REV5_OUT, index=False) # Was SAR_ERR_REV5_OUT
        # ------------------------

        print(f"Saved S4 Rev. 5 error bank results -> {S4_ERR_REV5_OUT}")
        display_summary_report(s4_err_rev5_results, "S4 (QUR+RRF) Error Bank (vs Rev. 5)")

    if not s1_err_rev4_gold.empty:
        print(f"\nEvaluating {len(s1_err_rev4_gold)} S4 Rev. 4 error bank queries...")
        for _, row in s1_err_rev4_gold.iterrows():
            q = row['question']
            cid = row['control_id']
            # (FIX) Safely get 'odp_required', default to "N/A"
            odp = str(row.get("odp_required", "N/A")).strip()

            final_rank = get_rrf_fused_rank(
                original_query=q,
                gold_cid=cid,
                bm25=bm25_rev4,
                control_ids=rev4_ids,
                qur_df=s4_errb_qur,
                top_n=TOP_N
            )

            # (FIX) Append a new dictionary, not the 'row' Series
            s4_err_rev4_records.append({
                "question": q,
                "control_id": cid,
                "odp_required": odp,
                "rank": final_rank,
                "version": "rev4" # Add version for the report
            })

        s4_err_rev4_results = pd.DataFrame(s4_err_rev4_records)
        s4_err_rev4_results.to_csv(S4_ERR_REV4_OUT, index=False)
        print(f"Saved S4 Rev. 4 error bank results -> {S4_ERR_REV4_OUT}")
        display_summary_report(s4_err_rev4_results, "S4 (QUR+RRF) Error Bank (vs Rev. 4)")

    # --- 6. Save Final S4 Report ---
    if not s4_rev5_results.empty and not s4_rev4_results.empty:
        print(f"\nSaving S4 summary report to {S4_SUMMARY_MD}...")
        save_summary_report(
            s4_rev5_results,
            s4_rev4_results,
            out_path=S4_SUMMARY_MD,
            system_name="System 4: BM25 + QUR (RRF)",
            err_rev5_df=s4_err_rev5_results,
            err_rev4_df=s4_err_rev4_results
        )
    else:
        print("\nCould not save S4 summary report. Main Rev5 or Rev4 results are missing.")

except KeyError as e:
    print(f"\n--- CRITICAL ERROR ---")
    print(f"A required column is missing from one of the CSVs: {e}")
except FileNotFoundError as e:
    print(f"\n--- CRITICAL ERROR ---")
    print(f"Could not find an input file: {e}")
    print("Please double-check the file paths in the S1 and S4/S3 Config cells.")
except Exception as e:
    print(f"\nAn unexpected error occurred in S4: {e}")

--- S4 (QUR+RRF) Evaluation ---
Loading S1 Ground Truth and S4 (QUR) Rewrites...

--- Running S4 (QUR+RRF) Rev. 5 Evaluation ---
Saved S4 Rev. 5 detailed results -> /content/drive/MyDrive/ablation_outputs/system_4_qur_rrf/S4_qur_rrf_rev5_results.csv

--- S4 (QUR+RRF) - NIST SP 800-53 Rev. 5: Performance ---
| Metric    | Overall (100q) | ODP-Subset (63q) |
|-----------|------------------|--------------------|
| Recall@1  | 0.6600           | 0.6349             |
| Recall@5  | 0.9700           | 1.0000             |
| Recall@10 | 0.9800           | 1.0000             |
| MRR@10    | 0.7802           | 0.7780             |
| nDCG@10   | 0.8300           | 0.8337             |

--- Running S4 (QUR+RRF) Rev. 4 Evaluation ---
Saved S4 Rev. 4 detailed results -> /content/drive/MyDrive/ablation_outputs/system_4_qur_rrf/S4_qur_rrf_rev4_results.csv

--- S4 (QUR+RRF) - NIST SP 800-53 Rev. 4: Performance ---
| Metric    | Overall (36q) | ODP-Subset (19q) |
|-----------|------------------|--------

## - **S5 (Hybrid RRF):**
Proved that fusing the two baselines (S1+S2) creates a superior high‑recall retriever that outperforms all previous systems on the main (Rev. 5) dataset.  


In [22]:
# ===================================================================
# --- SYSTEM 5: HYBRID RETRIEVAL (BM25 + DENSE) ---
# ===================================================================

print("--- Initializing S5 Configuration ---")

# --- 1. S5 (Hybrid RRF) Output Paths ---
S5_OUTPUT_DIR = "/content/drive/MyDrive/ablation_outputs/system_5_hybrid_rrf"
os.makedirs(S5_OUTPUT_DIR, exist_ok=True)
print(f"S5 Output directory: {S5_OUTPUT_DIR}")

S5_REV5_OUT = os.path.join(S5_OUTPUT_DIR, "S5_hybrid_rrf_rev5_results.csv")
S5_REV4_OUT = os.path.join(S5_OUTPUT_DIR, "S5_hybrid_rrf_rev4_results.csv")
S5_ERR_REV5_OUT = os.path.join(S5_OUTPUT_DIR, "S5_hybrid_rrf_error_bank_rev5_results.csv")
S5_ERR_REV4_OUT = os.path.join(S5_OUTPUT_DIR, "S5_hybrid_rrf_error_bank_rev4_results.csv")
S5_SUMMARY_MD = os.path.join(S5_OUTPUT_DIR, "S5_hybrid_rrf_report.md")

# --- 2. RRF Fusion Parameter ---
HYBRID_RRF_K = 60

--- Initializing S5 Configuration ---
S5 Output directory: /content/drive/MyDrive/ablation_outputs/system_5_hybrid_rrf


In [23]:
# --- S5 Helper Function (Hybrid RRF) ---

def get_hybrid_rrf_rank(
    query: str,
    gold_cid: str,
    bm25_retriever: BM25Okapi,
    dense_retriever: DenseRetriever,
    bm25_control_ids: List[str],
    top_n: int = 10,
    rrf_k: int = 60
) -> int:
    """
    Performs a Hybrid retrieval by fusing BM25 and Dense results.

    1. Runs a search on the S1 (BM25) retriever.
    2. Runs a search on the S2 (Dense) retriever.
    3. Fuses the two result lists using Reciprocal Rank Fusion (RRF).
    4. Finds the final rank of the gold_cid in the fused list.
    """

    # --- 1. Get S1 (BM25) results ---
    q_norm = normalize_text(query)
    q_tokens = tokenize(q_norm)
    bm25_idxs = bm25_retriever.get_top_n(q_tokens, n=top_n)
    bm25_results = [bm25_control_ids[i] for i in bm25_idxs]

    # --- 2. Get S2 (Dense) results ---
    try:
        dense_docs_scores = dense_retriever.search(query, top_k=top_n)
        dense_results = [str(doc_id).upper() for doc_id, score in dense_docs_scores]
    except Exception as e:
        print(f"  Warning: Dense retriever failed for query '{query[:30]}...': {e}")
        dense_results = []

    # --- 3. Apply Reciprocal Rank Fusion (RRF) ---
    rrf_scores = {}

    # Add BM25 scores
    for rank, cid in enumerate(bm25_results, start=1):
        rrf_score = 1.0 / (rrf_k + rank)
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + rrf_score

    # Add Dense scores
    for rank, cid in enumerate(dense_results, start=1):
        rrf_score = 1.0 / (rrf_k + rank)
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + rrf_score

    # --- 4. Get final fused ranking ---
    if not rrf_scores:
        return 0 # No results found at all

    # Sort by score, descending
    fused_ranking = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)
    fused_cids = [cid for cid, score in fused_ranking]

    # --- 5. Find rank of the gold_cid ---
    try:
        # 1-based rank
        final_rank = fused_cids.index(gold_cid) + 1
        return final_rank
    except ValueError:
        return 0 # Not found in the fused top results

In [24]:
# ===================================================================
# --- S5 MAIN: Run Hybrid (BM25 + Dense) Evaluation w/ RRF ---
# ===================================================================
# This re-uses S1 models (bm25_rev5, rev5_ids) and
# S2 models (dense_retriever_rev5)

print("--- S5 (Hybrid QUR+RRF) Evaluation ---")
print("Loading S1 Ground Truth Sets...")

# Check if S1 and S2 models are in memory
if 'bm25_rev5' not in locals() or 'dense_retriever_rev5' not in locals():
    print("="*50)
    print("CRITICAL ERROR: S1 and S2 models not found in memory.")
    print("Please re-run all S1 and S2 cells before running S5.")
    print("="*50)
else:
    try:
        # --- 1. Load S1 Ground Truth Sets ---
        s1_rev5_gold = pd.read_csv(REV5_GOLD_CSV, encoding="ISO-8859-1")
        s1_rev4_gold = pd.read_csv(REV4_GOLD_CSV, encoding="ISO-8859-1")
        s1_error_bank = pd.read_csv(ERROR_BANK_CSV, encoding="ISO-8859-1")

        # --- 3. Evaluate S5 Rev 5 (Hybrid RRF) ---
        print("\n--- Running S5 (Hybrid RRF) Rev. 5 Evaluation ---")
        s5_rev5_records = []

        for _, row in s1_rev5_gold.iterrows():
            q = row['question']
            cid = row['control_id']
            odp = str(row.get("odp_required", "N/A")).strip()

            final_rank = get_hybrid_rrf_rank(
                query=q,
                gold_cid=cid,
                bm25_retriever=bm25_rev5,
                dense_retriever=dense_retriever_rev5,
                bm25_control_ids=rev5_ids,
                top_n=TOP_N,
                rrf_k=HYBRID_RRF_K
            )

            s5_rev5_records.append({
                "question": q, "control_id": cid, "odp_required": odp, "rank": final_rank
            })

        s5_rev5_results = pd.DataFrame(s5_rev5_records)
        s5_rev5_results.to_csv(S5_REV5_OUT, index=False)
        print(f"Saved S5 Rev. 5 detailed results -> {S5_REV5_OUT}")
        display_summary_report(s5_rev5_results, "S5 (Hybrid RRF) - NIST SP 800-53 Rev. 5")

        # --- 4. Evaluate S5 Rev 4 (Hybrid RRF) ---
        print("\n--- Running S5 (Hybrid RRF) Rev. 4 Evaluation ---")
        s5_rev4_records = []

        for _, row in s1_rev4_gold.iterrows():
            q = row['question']
            cid = row['control_id']
            odp = str(row.get("odp_required", "N/A")).strip()

            final_rank = get_hybrid_rrf_rank(
                query=q,
                gold_cid=cid,
                bm25_retriever=bm25_rev4,
                dense_retriever=dense_retriever_rev4,
                bm25_control_ids=rev4_ids,
                top_n=TOP_N,
                rrf_k=HYBRID_RRF_K
            )

            s5_rev4_records.append({
                "question": q, "control_id": cid, "odp_required": odp, "rank": final_rank
            })

        s5_rev4_results = pd.DataFrame(s5_rev4_records)
        s5_rev4_results.to_csv(S5_REV4_OUT, index=False)
        print(f"Saved S5 Rev. 4 detailed results -> {S5_REV4_OUT}")
        display_summary_report(s5_rev4_results, "S5 (Hybrid RRF) - NIST SP 800-53 Rev. 4")

        # --- 5. Evaluate S5 Error Bank (Hybrid RRF) ---
        print("\n--- Running S5 (Hybrid RRF) Error Bank Evaluation ---")
        s5_err_rev5_records = []
        s5_err_rev4_records = []
        s5_err_rev5_results = pd.DataFrame() # Init empty
        s5_err_rev4_results = pd.DataFrame() # Init empty

        s1_error_bank['version'] = s1_error_bank['version'].astype(str).str.lower().str.strip()
        s1_err_rev5_gold = s1_error_bank[s1_error_bank['version'] == 'rev5']
        s1_err_rev4_gold = s1_error_bank[s1_error_bank['version'] == 'rev4']

        if not s1_err_rev5_gold.empty:
            print(f"\nEvaluating {len(s1_err_rev5_gold)} S5 Rev. 5 error bank queries...")
            for _, row in s1_err_rev5_gold.iterrows():
                q = row['question']
                cid = row['control_id']
                odp = str(row.get("odp_required", "N/A")).strip()

                final_rank = get_hybrid_rrf_rank(
                    query=q,
                    gold_cid=cid,
                    bm25_retriever=bm25_rev5,
                    dense_retriever=dense_retriever_rev5,
                    bm25_control_ids=rev5_ids,
                    top_n=TOP_N,
                    rrf_k=HYBRID_RRF_K
                )

                s5_err_rev5_records.append({
                    "question": q, "control_id": cid, "odp_required": odp, "rank": final_rank, "version": "rev5"
                })

            s5_err_rev5_results = pd.DataFrame(s5_err_rev5_records)
            s5_err_rev5_results.to_csv(S5_ERR_REV5_OUT, index=False)
            print(f"Saved S5 Rev. 5 error bank results -> {S5_ERR_REV5_OUT}")
            display_summary_report(s5_err_rev5_results, "S5 (Hybrid RRF) Error Bank (vs Rev. 5)")

        if not s1_err_rev4_gold.empty:
            print(f"\nEvaluating {len(s1_err_rev4_gold)} S5 Rev. 4 error bank queries...")
            for _, row in s1_err_rev4_gold.iterrows():
                q = row['question']
                cid = row['control_id']
                odp = str(row.get("odp_required", "N/A")).strip()

                final_rank = get_hybrid_rrf_rank(
                    query=q,
                    gold_cid=cid,
                    bm25_retriever=bm25_rev4,
                    dense_retriever=dense_retriever_rev4,
                    bm25_control_ids=rev4_ids,
                    top_n=TOP_N,
                    rrf_k=HYBRID_RRF_K
                )

                s5_err_rev4_records.append({
                    "question": q, "control_id": cid, "odp_required": odp, "rank": final_rank, "version": "rev4"
                })

            s5_err_rev4_results = pd.DataFrame(s5_err_rev4_records)
            s5_err_rev4_results.to_csv(S5_ERR_REV4_OUT, index=False)
            print(f"Saved S5 Rev. 4 error bank results -> {S5_ERR_REV4_OUT}")
            display_summary_report(s5_err_rev4_results, "S5 (Hybrid RRF) Error Bank (vs Rev. 4)")

        # --- 6. Save Final S5 Report ---
        if not s5_rev5_results.empty and not s5_rev4_results.empty:
            print(f"\nSaving S5 summary report to {S5_SUMMARY_MD}...")
            save_summary_report(
                s5_rev5_results,
                s5_rev4_results,
                out_path=S5_SUMMARY_MD,
                system_name="System 5: Hybrid (BM25+Dense) RRF",
                err_rev5_df=s5_err_rev5_results,
                err_rev4_df=s5_err_rev4_results
            )
        else:
            print("\nCould not save S5 summary report. Main Rev5 or Rev4 results are missing.")

    except Exception as e:
        print(f"\nAn unexpected error occurred in S5: {e}")

--- S5 (Hybrid QUR+RRF) Evaluation ---
Loading S1 Ground Truth Sets...

--- Running S5 (Hybrid RRF) Rev. 5 Evaluation ---
Saved S5 Rev. 5 detailed results -> /content/drive/MyDrive/ablation_outputs/system_5_hybrid_rrf/S5_hybrid_rrf_rev5_results.csv

--- S5 (Hybrid RRF) - NIST SP 800-53 Rev. 5: Performance ---
| Metric    | Overall (100q) | ODP-Subset (63q) |
|-----------|------------------|--------------------|
| Recall@1  | 0.8500           | 0.8730             |
| Recall@5  | 1.0000           | 1.0000             |
| Recall@10 | 1.0000           | 1.0000             |
| MRR@10    | 0.9153           | 0.9286             |
| nDCG@10   | 0.9369           | 0.9468             |

--- Running S5 (Hybrid RRF) Rev. 4 Evaluation ---
Saved S5 Rev. 4 detailed results -> /content/drive/MyDrive/ablation_outputs/system_5_hybrid_rrf/S5_hybrid_rrf_rev4_results.csv

--- S5 (Hybrid RRF) - NIST SP 800-53 Rev. 4: Performance ---
| Metric    | Overall (36q) | ODP-Subset (19q) |
|-----------|-------------

## - **S6 (Hybrid + Rerank):**
Proved that adding a reranker to S5’s high‑recall list is the key to precision, solving the “noise problem” from S5 and achieving state‑of‑the‑art results on the Error Bank.  

In [25]:
# ===================================================================
# --- SYSTEM 6: HYBRID + RERANKER (BM25 + DENSE + RERANK) ---
# ===================================================================

print("--- Initializing S6 Configuration ---")

# --- 1. S6 (Hybrid + Reranker) Output Paths ---
S6_OUTPUT_DIR = "/content/drive/MyDrive/ablation_outputs/system_6_hybrid_rerank"
os.makedirs(S6_OUTPUT_DIR, exist_ok=True)
# (FIXED TYPO: S6_OUTPUT_DOT -> S6_OUTPUT_DIR)
print(f"S6 Output directory: {S6_OUTPUT_DIR}")

S6_REV5_OUT = os.path.join(S6_OUTPUT_DIR, "S6_hybrid_rerank_rev5_results.csv")
S6_REV4_OUT = os.path.join(S6_OUTPUT_DIR, "S6_hybrid_rerank_rev4_results.csv")
S6_ERR_REV5_OUT = os.path.join(S6_OUTPUT_DIR, "S6_hybrid_rerank_error_bank_rev5_results.csv")
S6_ERR_REV4_OUT = os.path.join(S6_OUTPUT_DIR, "S6_hybrid_rerank_error_bank_rev4_results.csv")
S6_SUMMARY_MD = os.path.join(S6_OUTPUT_DIR, "S6_hybrid_rerank_report.md")

# --- 2. Reranker Parameters ---
RERANKER_MODEL_ID = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
# How many top RRF results to feed into the reranker
CANDIDATE_SET_SIZE = 50
# RRF-specific parameter (from S5)
HYBRID_RRF_K = 60

--- Initializing S6 Configuration ---
S6 Output directory: /content/drive/MyDrive/ablation_outputs/system_6_hybrid_rerank


In [26]:
# --- S6 Helper Function (Hybrid RRF + Reranker) ---

from sentence_transformers import CrossEncoder

def get_hybrid_reranked_rank(
    query: str,
    gold_cid: str,
    bm25_retriever: BM25Okapi,
    dense_retriever: DenseRetriever,
    bm25_control_ids: List[str],
    cross_enc: CrossEncoder,
    id_to_text_map: Dict[str, str], # We need full text for the reranker
    candidate_set_size: int = 50, # Use a larger candidate set
    top_n: int = 10, # The final metric cutoff (e.g., @10)
    rrf_k: int = 60
) -> int:
    """
    Performs a 2-stage retrieval:
    1. First stage: Hybrid RRF (BM25+Dense) to get a high-recall candidate set.
    2. Second stage: Cross-encoder reranks the candidate set for high precision.
    """

    # --- STAGE 1: Get Hybrid RRF Candidate Set ---

    # Get S1 (BM25) results
    q_norm = normalize_text(query)
    q_tokens = tokenize(q_norm)
    bm25_idxs = bm25_retriever.get_top_n(q_tokens, n=candidate_set_size)
    bm25_results = [bm25_control_ids[i] for i in bm25_idxs]

    # Get S2 (Dense) results
    try:
        dense_docs_scores = dense_retriever.search(query, top_k=candidate_set_size)
        dense_results = [str(doc_id).upper() for doc_id, score in dense_docs_scores]
    except Exception as e:
        dense_results = []

    # Apply RRF
    rrf_scores = {}
    for rank, cid in enumerate(bm25_results, start=1):
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (1.0 / (rrf_k + rank))
    for rank, cid in enumerate(dense_results, start=1):
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (1.0 / (rrf_k + rank))

    if not rrf_scores:
        return 0 # No results found at all

    # Get top N candidates from RRF
    fused_ranking = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)
    candidate_cids = [cid for cid, score in fused_ranking[:candidate_set_size]]

    # --- STAGE 2: Rerank with Cross-Encoder ---

    # Create (query, passage) pairs for the reranker
    rerank_pairs = []
    # Keep track of which cids we are reranking
    rerank_cids_ordered = []

    for cid in candidate_cids:
        passage_text = id_to_text_map.get(cid)
        if passage_text: # Make sure we have text for this ID
            rerank_pairs.append((query, passage_text))
            rerank_cids_ordered.append(cid)

    if not rerank_pairs:
        return 0 # No valid candidates to rerank

    # Get scores from the cross-encoder
    # model.predict() returns a list of scores in the same order as the input
    rerank_scores = cross_enc.predict(rerank_pairs, show_progress_bar=False)

    # Combine CIDs with their new scores
    reranked_results = list(zip(rerank_cids_ordered, rerank_scores))

    # Sort by the new score, descending
    reranked_results.sort(key=lambda item: item[1], reverse=True)

    # Get the final list of CIDs
    final_ranked_cids = [cid for cid, score in reranked_results]

    # --- 3. Find rank of the gold_cid in the FINAL list ---
    try:
        # 1-based rank
        final_rank = final_ranked_cids.index(gold_cid) + 1

        # Only count if it's in the top_n (e.g., Top 10)
        if final_rank <= top_n:
            return final_rank
        else:
            return 0 # Found, but not in Top 10
    except ValueError:
        return 0 # Not found in the reranked list

In [27]:
# ===================================================================
# --- S6 MAIN: Run Hybrid + Reranker Evaluation ---
# ===================================================================
# This re-uses S1 models (bm25_rev5, rev5_docs) and
# S2 models (dense_retriever_rev5)

print("--- S6 (Hybrid + Reranker) Evaluation ---")

# Check if S1/S2 models are in memory
if 'bm25_rev5' not in locals() or 'dense_retriever_rev5' not in locals() or 'rev5_docs' not in locals():
    print("="*50)
    print("CRITICAL ERROR: S1 and S2 models/data not found in memory.")
    print("Please re-run all S1 (cell 10) and S2 (cell 16) MAIN cells before running S6.")
    print("="*50)
else:
    try:
        # --- 1. Initialize Cross-Encoder Model ---
        print(f"Loading reranker model: {RERANKER_MODEL_ID}...")
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        cross_encoder = CrossEncoder(RERANKER_MODEL_ID, max_length=512, device=device)
        print(f"Reranker model loaded on {device}.")

        # --- 2. Create Control_ID -> Text Maps (CRITICAL for Reranking) ---
        # We get rev5_docs and rev4_docs from S1 (Cell 10)
        print("Building Control ID to Text maps...")
        rev5_id_to_text = {doc['control_id']: doc['text'] for doc in rev5_docs}
        rev4_id_to_text = {doc['control_id']: doc['text'] for doc in rev4_docs}
        print(f"Maps built: {len(rev5_id_to_text)} Rev. 5, {len(rev4_id_to_text)} Rev. 4 entries.")

        # --- 3. Load S1 Ground Truth Sets ---
        print("Loading S1 Ground Truth Sets...")
        s1_rev5_gold = pd.read_csv(REV5_GOLD_CSV, encoding="ISO-8859-1")
        s1_rev4_gold = pd.read_csv(REV4_GOLD_CSV, encoding="ISO-8859-1")
        s1_error_bank = pd.read_csv(ERROR_BANK_CSV, encoding="ISO-8859-1")

        # --- 4. Evaluate S6 Rev 5 (Hybrid + Rerank) ---
        print("\n--- Running S6 (Hybrid + Rerank) Rev. 5 Evaluation ---")
        s6_rev5_records = []

        for i, row in s1_rev5_gold.iterrows():
            if (i + 1) % 25 == 0:
                print(f"  Processing query {i+1}/{len(s1_rev5_gold)}...")
            q = row['question']
            cid = row['control_id']
            odp = str(row.get("odp_required", "N/A")).strip()

            final_rank = get_hybrid_reranked_rank(
                query=q,
                gold_cid=cid,
                bm25_retriever=bm25_rev5,
                dense_retriever=dense_retriever_rev5,
                bm25_control_ids=rev5_ids,
                cross_enc=cross_encoder,
                id_to_text_map=rev5_id_to_text,
                candidate_set_size=CANDIDATE_SET_SIZE,
                top_n=TOP_N,
                rrf_k=HYBRID_RRF_K
            )

            s6_rev5_records.append({
                "question": q, "control_id": cid, "odp_required": odp, "rank": final_rank
            })

        s6_rev5_results = pd.DataFrame(s6_rev5_records)
        s6_rev5_results.to_csv(S6_REV5_OUT, index=False)
        print(f"Saved S6 Rev. 5 detailed results -> {S6_REV5_OUT}")
        display_summary_report(s6_rev5_results, "S6 (Hybrid + Rerank) - NIST SP 800-53 Rev. 5")

        # --- 5. Evaluate S6 Rev 4 (Hybrid + Rerank) ---
        print("\n--- Running S6 (Hybrid + Rerank) Rev. 4 Evaluation ---")
        s6_rev4_records = []

        for i, row in s1_rev4_gold.iterrows():
            if (i + 1) % 10 == 0:
                 print(f"  Processing query {i+1}/{len(s1_rev4_gold)}...")
            q = row['question']
            cid = row['control_id']
            odp = str(row.get("odp_required", "N/A")).strip()

            final_rank = get_hybrid_reranked_rank(
                query=q,
                gold_cid=cid,
                bm25_retriever=bm25_rev4,
                dense_retriever=dense_retriever_rev4,
                bm25_control_ids=rev4_ids,
                cross_enc=cross_encoder,
                id_to_text_map=rev4_id_to_text,
                candidate_set_size=CANDIDATE_SET_SIZE,
                top_n=TOP_N,
                rrf_k=HYBRID_RRF_K
            )

            s6_rev4_records.append({
                "question": q, "control_id": cid, "odp_required": odp, "rank": final_rank
            })

        s6_rev4_results = pd.DataFrame(s6_rev4_records)
        s6_rev4_results.to_csv(S6_REV4_OUT, index=False)
        print(f"Saved S6 Rev. 4 detailed results -> {S6_REV4_OUT}")
        display_summary_report(s6_rev4_results, "S6 (Hybrid + Rerank) - NIST SP 800-53 Rev. 4")

        # --- 6. Evaluate S6 Error Bank (Hybrid + Rerank) ---
        print("\n--- Running S6 (Hybrid + Rerank) Error Bank Evaluation ---")
        s6_err_rev5_records = []
        s6_err_rev4_records = []
        s6_err_rev5_results = pd.DataFrame() # Init empty
        s6_err_rev4_results = pd.DataFrame() # Init empty

        s1_error_bank['version'] = s1_error_bank['version'].astype(str).str.lower().str.strip()
        s1_err_rev5_gold = s1_error_bank[s1_error_bank['version'] == 'rev5']
        s1_err_rev4_gold = s1_error_bank[s1_error_bank['version'] == 'rev4']

        if not s1_err_rev5_gold.empty:
            print(f"\nEvaluating {len(s1_err_rev5_gold)} S6 Rev. 5 error bank queries...")
            for i, row in s1_err_rev5_gold.iterrows():
                if (i + 1) % 10 == 0:
                    print(f"  Processing query {i+1}/{len(s1_err_rev5_gold)}...")
                q = row['question']
                cid = row['control_id']
                odp = str(row.get("odp_required", "N/A")).strip()

                final_rank = get_hybrid_reranked_rank(
                    query=q,
                    gold_cid=cid,
                    bm25_retriever=bm25_rev5,
                    dense_retriever=dense_retriever_rev5,
                    bm25_control_ids=rev5_ids,
                    cross_enc=cross_encoder,
                    id_to_text_map=rev5_id_to_text,
                    candidate_set_size=CANDIDATE_SET_SIZE,
                    top_n=TOP_N,
                    rrf_k=HYBRID_RRF_K
                )

                s6_err_rev5_records.append({
                    "question": q, "control_id": cid, "odp_required": odp, "rank": final_rank, "version": "rev5"
                })

            s6_err_rev5_results = pd.DataFrame(s6_err_rev5_records)
            s6_err_rev5_results.to_csv(S6_ERR_REV5_OUT, index=False)
            print(f"Saved S6 Rev. 5 error bank results -> {S6_ERR_REV5_OUT}")
            display_summary_report(s6_err_rev5_results, "S6 (Hybrid + Rerank) Error Bank (vs Rev. 5)")

        if not s1_err_rev4_gold.empty:
            print(f"\nEvaluating {len(s1_err_rev4_gold)} S6 Rev. 4 error bank queries...")
            for i, row in s1_err_rev4_gold.iterrows():
                if (i + 1) % 10 == 0:
                    print(f"  Processing query {i+1}/{len(s1_err_rev4_gold)}...")
                q = row['question']
                cid = row['control_id']
                odp = str(row.get("odp_required", "N/A")).strip()

                final_rank = get_hybrid_reranked_rank(
                    query=q,
                    gold_cid=cid,
                    bm25_retriever=bm25_rev4,
                    dense_retriever=dense_retriever_rev4,
                    bm25_control_ids=rev4_ids,
                    cross_enc=cross_encoder,
                    id_to_text_map=rev4_id_to_text,
                    candidate_set_size=CANDIDATE_SET_SIZE,
                    top_n=TOP_N,
                    rrf_k=HYBRID_RRF_K
                )

                s6_err_rev4_records.append({
                    "question": q, "control_id": cid, "odp_required": odp, "rank": final_rank, "version": "rev4"
                })

            s6_err_rev4_results = pd.DataFrame(s6_err_rev4_records)
            s6_err_rev4_results.to_csv(S6_ERR_REV4_OUT, index=False)
            print(f"Saved S6 Rev. 4 error bank results -> {S6_ERR_REV4_OUT}")
            display_summary_report(s6_err_rev4_results, "S6 (Hybrid + Rerank) Error Bank (vs Rev. 4)")

        # --- 7. Save Final S6 Report ---
        if not s6_rev5_results.empty and not s6_rev4_results.empty:
            print(f"\nSaving S6 summary report to {S6_SUMMARY_MD}...")
            save_summary_report(
                s6_rev5_results,
                s6_rev4_results,
                out_path=S6_SUMMARY_MD,
                system_name="System 6: Hybrid + Reranker",
                err_rev5_df=s6_err_rev5_results,
                err_rev4_df=s6_err_rev4_results
            )
        else:
            print("\nCould not save S6 summary report. Main Rev5 or Rev4 results are missing.")

    except Exception as e:
        print(f"\nAn unexpected error occurred in S6: {e}")

--- S6 (Hybrid + Reranker) Evaluation ---
Loading reranker model: cross-encoder/ms-marco-MiniLM-L-6-v2...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Reranker model loaded on cuda.
Building Control ID to Text maps...
Maps built: 1193 Rev. 5, 922 Rev. 4 entries.
Loading S1 Ground Truth Sets...

--- Running S6 (Hybrid + Rerank) Rev. 5 Evaluation ---
  Processing query 25/100...
  Processing query 50/100...
  Processing query 75/100...
  Processing query 100/100...
Saved S6 Rev. 5 detailed results -> /content/drive/MyDrive/ablation_outputs/system_6_hybrid_rerank/S6_hybrid_rerank_rev5_results.csv

--- S6 (Hybrid + Rerank) - NIST SP 800-53 Rev. 5: Performance ---
| Metric    | Overall (100q) | ODP-Subset (63q) |
|-----------|------------------|--------------------|
| Recall@1  | 0.8900           | 0.9048             |
| Recall@5  | 0.9900           | 1.0000             |
| Recall@10 | 1.0000           | 1.0000             |
| MRR@10    | 0.9398           | 0.9497             |
| nDCG@10   | 0.9551           | 0.9628             |

--- Running S6 (Hybrid + Rerank) Rev. 4 Evaluation ---
  Processing query 10/36...
  Processing query 20/36.

## - **S7 (ComplianceGPT):**
Proved that the full architecture (QUR + Hybrid + Reranker) is the optimal system, as the QUR component provides a final, measurable boost on top of S6, especially on the Error Bank.

In [28]:
# ===================================================================
# --- SYSTEM 7: ComplianceGPT (QUR + HYBRID RRF + RERANKER) ---
# ===================================================================

print("--- Initializing S7 (ComplianceGPT) Configuration ---")

# --- 1. S7 (ComplianceGPT) Output Paths ---
S7_OUTPUT_DIR = "/content/drive/MyDrive/ablation_outputs/system_7_compliance_gpt"
os.makedirs(S7_OUTPUT_DIR, exist_ok=True)
print(f"S7 Output directory: {S7_OUTPUT_DIR}")

S7_REV5_OUT = os.path.join(S7_OUTPUT_DIR, "S7_compliance_gpt_rev5_results.csv")
S7_REV4_OUT = os.path.join(S7_OUTPUT_DIR, "S7_compliance_gpt_rev4_results.csv")
S7_ERR_REV5_OUT = os.path.join(S7_OUTPUT_DIR, "S7_compliance_gpt_error_bank_rev5_results.csv")
S7_ERR_REV4_OUT = os.path.join(S7_OUTPUT_DIR, "S7_compliance_gpt_error_bank_rev4_results.csv")
S7_SUMMARY_MD = os.path.join(S7_OUTPUT_DIR, "S7_compliance_gpt_report.md")

# --- 2. Reranker/RRF Parameters (re-using from S6/S5) ---
# RERANKER_MODEL_ID is already in memory from S6
# CANDIDATE_SET_SIZE is already in memory from S6
# HYBRID_RRF_K is already in memory from S6

--- Initializing S7 (ComplianceGPT) Configuration ---
S7 Output directory: /content/drive/MyDrive/ablation_outputs/system_7_compliance_gpt


In [29]:
# --- S7 Helper Function (ComplianceGPT) ---

def get_compliance_gpt_rank(
    original_query: str,
    gold_cid: str,
    bm25_retriever: BM25Okapi,
    dense_retriever: DenseRetriever,
    bm25_control_ids: List[str],
    cross_enc: CrossEncoder,
    id_to_text_map: Dict[str, str],
    qur_df: pd.DataFrame, # For getting rewrites
    candidate_set_size: int = 50,
    top_n: int = 10,
    rrf_k: int = 60
) -> int:
    """
    Performs the full 2-stage ComplianceGPT retrieval:
    1. First stage: "Superhybrid" RRF
       - Fuses S4 (BM25 + all QUR rewrites)
       - Fuses S2 (Dense on original query)
    2. Second stage: Cross-encoder reranks the candidate set.
    """

    # --- STAGE 1: Get "Superhybrid" RRF Candidate Set ---

    rrf_scores = {}

    # --- Part A: Get S4 (QUR-RRF) results ---

    # 1. Get all queries (original + rewrites)
    q_cleaned = original_query.strip().lstrip(": ")
    matching_rows = qur_df[qur_df['original_query'] == q_cleaned]
    queries_to_run_bm25 = [original_query] # Start with the original
    if not matching_rows.empty:
        queries_to_run_bm25.extend(matching_rows['rewritten_query'].astype(str).str.strip().str.lstrip(": ").tolist())
    queries_to_run_bm25 = list(dict.fromkeys(queries_to_run_bm25))

    # 2. Run BM25 for all queries and add to RRF scores
    for q in queries_to_run_bm25:
        q_norm = normalize_text(q)
        q_tokens = tokenize(q_norm)
        bm25_idxs = bm25_retriever.get_top_n(q_tokens, n=candidate_set_size)
        bm25_results = [bm25_control_ids[i] for i in bm25_idxs]
        for rank, cid in enumerate(bm25_results, start=1):
            rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (1.0 / (rrf_k + rank))

    # --- Part B: Get S2 (Dense) results ---
    try:
        dense_docs_scores = dense_retriever.search(original_query, top_k=candidate_set_size)
        dense_results = [str(doc_id).upper() for doc_id, score in dense_docs_scores]
        # 3. Add Dense results to RRF scores
        for rank, cid in enumerate(dense_results, start=1):
            rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (1.0 / (rrf_k + rank))
    except Exception as e:
        print(f"  Warning: Dense retriever failed for query '{original_query[:30]}...': {e}")

    if not rrf_scores:
        return 0 # No results found at all

    # 4. Get top N candidates from the "Superhybrid" RRF pool
    fused_ranking = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)
    candidate_cids = [cid for cid, score in fused_ranking[:candidate_set_size]]

    # --- STAGE 2: Rerank with Cross-Encoder (Identical to S6) ---

    rerank_pairs = []
    rerank_cids_ordered = []

    for cid in candidate_cids:
        passage_text = id_to_text_map.get(cid)
        if passage_text:
            rerank_pairs.append((original_query, passage_text))
            rerank_cids_ordered.append(cid)

    if not rerank_pairs:
        return 0

    rerank_scores = cross_enc.predict(rerank_pairs, show_progress_bar=False)
    reranked_results = list(zip(rerank_cids_ordered, rerank_scores))
    reranked_results.sort(key=lambda item: item[1], reverse=True)
    final_ranked_cids = [cid for cid, score in reranked_results]

    # --- 3. Find rank of the gold_cid in the FINAL list ---
    try:
        final_rank = final_ranked_cids.index(gold_cid) + 1
        if final_rank <= top_n:
            return final_rank
        else:
            return 0 # Found, but not in Top 10
    except ValueError:
        return 0 # Not found in the reranked list

In [30]:
# ===================================================================
# --- S7 MAIN: Run ComplianceGPT (QUR + Hybrid + Rerank) Evaluation ---
# ===================================================================
# This re-uses all S1, S2, and S6 models and data.

print("--- S7 (ComplianceGPT) Evaluation ---")

# Check if all models are in memory
if 'bm25_rev5' not in locals() or 'dense_retriever_rev5' not in locals() or 'cross_encoder' not in locals() or 'rev5_id_to_text' not in locals():
    print("="*50)
    print("CRITICAL ERROR: S1, S2, or S6 models/data not found in memory.")
    print("Please re-run all S1, S2, and S6 cells before running S7.")
    print("="*50)
else:
    try:
        # --- 1. Load S1 Ground Truth Sets ---
        print("Loading S1 Ground Truth Sets...")
        s1_rev5_gold = pd.read_csv(REV5_GOLD_CSV, encoding="ISO-8859-1")
        s1_rev4_gold = pd.read_csv(REV4_GOLD_CSV, encoding="ISO-8859-1")
        s1_error_bank = pd.read_csv(ERROR_BANK_CSV, encoding="ISO-8859-1")

        # --- 2. Load QUR Rewrite Sets (for S4 logic) ---
        print("Loading QUR Rewrite Sets...")
        s7_rev5_qur = pd.read_csv(QUR_CSV_REV5, encoding="ISO-8859-1")
        s7_rev4_qur = pd.read_csv(QUR_CSV_REV4, encoding="ISO-8859-1")
        s7_errb_qur = pd.read_csv(QUR_CSV_ERRB, encoding="ISO-8859-1")

        # Clean the join column in the rewrite files ONCE
        for df in [s7_rev5_qur, s7_rev4_qur, s7_errb_qur]:
            if 'original_query' in df.columns:
                df['original_query'] = df['original_query'].astype(str).str.strip().str.lstrip(": ")

        # --- 3. Evaluate S7 Rev 5 (ComplianceGPT) ---
        print("\n--- Running S7 (ComplianceGPT) Rev. 5 Evaluation ---")
        s7_rev5_records = []

        for i, row in s1_rev5_gold.iterrows():
            if (i + 1) % 25 == 0:
                print(f"  Processing query {i+1}/{len(s1_rev5_gold)}...")
            q = row['question']
            cid = row['control_id']
            odp = str(row.get("odp_required", "N/A")).strip()

            final_rank = get_compliance_gpt_rank(
                original_query=q,
                gold_cid=cid,
                bm25_retriever=bm25_rev5,
                dense_retriever=dense_retriever_rev5,
                bm25_control_ids=rev5_ids,
                cross_enc=cross_encoder,
                id_to_text_map=rev5_id_to_text,
                qur_df=s7_rev5_qur, # Pass in the rewrites
                candidate_set_size=CANDIDATE_SET_SIZE,
                top_n=TOP_N,
                rrf_k=HYBRID_RRF_K
            )

            s7_rev5_records.append({
                "question": q, "control_id": cid, "odp_required": odp, "rank": final_rank
            })

        s7_rev5_results = pd.DataFrame(s7_rev5_records)
        s7_rev5_results.to_csv(S7_REV5_OUT, index=False)
        print(f"Saved S7 Rev. 5 detailed results -> {S7_REV5_OUT}")
        display_summary_report(s7_rev5_results, "S7 (ComplianceGPT) - NIST SP 800-53 Rev. 5")

        # --- 4. Evaluate S7 Rev 4 (ComplianceGPT) ---
        print("\n--- Running S7 (ComplianceGPT) Rev. 4 Evaluation ---")
        s7_rev4_records = []

        for i, row in s1_rev4_gold.iterrows():
            if (i + 1) % 10 == 0:
                 print(f"  Processing query {i+1}/{len(s1_rev4_gold)}...")
            q = row['question']
            cid = row['control_id']
            odp = str(row.get("odp_required", "N/A")).strip()

            final_rank = get_compliance_gpt_rank(
                original_query=q,
                gold_cid=cid,
                bm25_retriever=bm25_rev4,
                dense_retriever=dense_retriever_rev4,
                bm25_control_ids=rev4_ids,
                cross_enc=cross_encoder,
                id_to_text_map=rev4_id_to_text,
                qur_df=s7_rev4_qur, # Pass in the rewrites
                candidate_set_size=CANDIDATE_SET_SIZE,
                top_n=TOP_N,
                rrf_k=HYBRID_RRF_K
            )

            s7_rev4_records.append({
                "question": q, "control_id": cid, "odp_required": odp, "rank": final_rank
            })

        s7_rev4_results = pd.DataFrame(s7_rev4_records)
        s7_rev4_results.to_csv(S7_REV4_OUT, index=False)
        print(f"Saved S7 Rev. 4 detailed results -> {S7_REV4_OUT}")
        display_summary_report(s7_rev4_results, "S7 (ComplianceGPT) - NIST SP 800-53 Rev. 4")

        # --- 5. Evaluate S7 Error Bank (ComplianceGPT) ---
        print("\n--- Running S7 (ComplianceGPT) Error Bank Evaluation ---")
        s7_err_rev5_records = []
        s7_err_rev4_records = []
        s7_err_rev5_results = pd.DataFrame() # Init empty
        s7_err_rev4_results = pd.DataFrame() # Init empty

        s1_error_bank['version'] = s1_error_bank['version'].astype(str).str.lower().str.strip()
        s1_err_rev5_gold = s1_error_bank[s1_error_bank['version'] == 'rev5']
        s1_err_rev4_gold = s1_error_bank[s1_error_bank['version'] == 'rev4']

        if not s1_err_rev5_gold.empty:
            print(f"\nEvaluating {len(s1_err_rev5_gold)} S7 Rev. 5 error bank queries...")
            for i, row in s1_err_rev5_gold.iterrows():
                if (i + 1) % 10 == 0:
                    print(f"  Processing query {i+1}/{len(s1_err_rev5_gold)}...")
                q = row['question']
                cid = row['control_id']
                odp = str(row.get("odp_required", "N/A")).strip()

                final_rank = get_compliance_gpt_rank(
                    original_query=q,
                    gold_cid=cid,
                    bm25_retriever=bm25_rev5,
                    dense_retriever=dense_retriever_rev5,
                    bm25_control_ids=rev5_ids,
                    cross_enc=cross_encoder,
                    id_to_text_map=rev5_id_to_text,
                    qur_df=s7_errb_qur, # Pass in the rewrites
                    candidate_set_size=CANDIDATE_SET_SIZE,
                    top_n=TOP_N,
                    rrf_k=HYBRID_RRF_K
                )

                s7_err_rev5_records.append({
                    "question": q, "control_id": cid, "odp_required": odp, "rank": final_rank, "version": "rev5"
                })

            s7_err_rev5_results = pd.DataFrame(s7_err_rev5_records)
            s7_err_rev5_results.to_csv(S7_ERR_REV5_OUT, index=False)
            print(f"Saved S7 Rev. 5 error bank results -> {S7_ERR_REV5_OUT}")
            display_summary_report(s7_err_rev5_results, "S7 (ComplianceGPT) Error Bank (vs Rev. 5)")

        if not s1_err_rev4_gold.empty:
            print(f"\nEvaluating {len(s1_err_rev4_gold)} S7 Rev. 4 error bank queries...")
            for i, row in s1_err_rev4_gold.iterrows():
                if (i + 1) % 10 == 0:
                    print(f"  Processing query {i+1}/{len(s1_err_rev4_gold)}...")
                q = row['question']
                cid = row['control_id']
                odp = str(row.get("odp_required", "N/A")).strip()

                final_rank = get_compliance_gpt_rank(
                    original_query=q,
                    gold_cid=cid,
                    bm25_retriever=bm25_rev4,
                    dense_retriever=dense_retriever_rev4,
                    bm25_control_ids=rev4_ids,
                    cross_enc=cross_encoder,
                    id_to_text_map=rev4_id_to_text,
                    qur_df=s7_errb_qur, # Pass in the rewrites
                    candidate_set_size=CANDIDATE_SET_SIZE,
                    top_n=TOP_N,
                    rrf_k=HYBRID_RRF_K
                )

                s7_err_rev4_records.append({
                    "question": q, "control_id": cid, "odp_required": odp, "rank": final_rank, "version": "rev4"
                })

            s7_err_rev4_results = pd.DataFrame(s7_err_rev4_records)
            s7_err_rev4_results.to_csv(S7_ERR_REV4_OUT, index=False)
            print(f"Saved S7 Rev. 4 error bank results -> {S7_ERR_REV4_OUT}")
            display_summary_report(s7_err_rev4_results, "S7 (ComplianceGPT) Error Bank (vs Rev. 4)")

        # --- 7. Save Final S7 Report ---
        if not s7_rev5_results.empty and not s7_rev4_results.empty:
            print(f"\nSaving S7 summary report to {S7_SUMMARY_MD}...")
            save_summary_report(
                s7_rev5_results,
                s7_rev4_results,
                out_path=S7_SUMMARY_MD,
                system_name="System 7: ComplianceGPT (Ultimate)",
                err_rev5_df=s7_err_rev5_results,
                err_rev4_df=s7_err_rev4_results
            )
        else:
            print("\nCould not save S7 summary report. Main Rev5 or Rev4 results are missing.")

    except Exception as e:
        print(f"\nAn unexpected error occurred in S7: {e}")

--- S7 (ComplianceGPT) Evaluation ---
Loading S1 Ground Truth Sets...
Loading QUR Rewrite Sets...

--- Running S7 (ComplianceGPT) Rev. 5 Evaluation ---
  Processing query 25/100...
  Processing query 50/100...
  Processing query 75/100...
  Processing query 100/100...
Saved S7 Rev. 5 detailed results -> /content/drive/MyDrive/ablation_outputs/system_7_compliance_gpt/S7_compliance_gpt_rev5_results.csv

--- S7 (ComplianceGPT) - NIST SP 800-53 Rev. 5: Performance ---
| Metric    | Overall (100q) | ODP-Subset (63q) |
|-----------|------------------|--------------------|
| Recall@1  | 0.9000           | 0.9206             |
| Recall@5  | 0.9900           | 1.0000             |
| Recall@10 | 1.0000           | 1.0000             |
| MRR@10    | 0.9448           | 0.9577             |
| nDCG@10   | 0.9588           | 0.9686             |

--- Running S7 (ComplianceGPT) Rev. 4 Evaluation ---
  Processing query 10/36...
  Processing query 20/36...
  Processing query 30/36...
Saved S7 Rev. 4 det